# NATM Results Analysis

Interactive exploration across all three sectors and all 8 policy/demand
variants: technology diffusion over time, energy demand by carrier over
time (with a segment filter), energy demand by port/hub, the overall fleet
transition summary, financial analysis (capex, NPV, and risk-adjusted
decision metrics from the ambiguity-aware framework), and (when generated)
a real-shape hourly demand profile.

Every chart below carries **Sector** and **Case** dropdowns (Aviation Cargo
/ Maritime Cargo / Aviation Passenger x the 8 policy/demand variants) so you
can switch both interactively without editing code or re-running cells.

For side-by-side comparison across all 8 variants at once (rather than
picking one at a time), see the "Compare all cases" section further down.

In [1]:
from __future__ import annotations

from pathlib import Path

import altair as alt
import pandas as pd

alt.data_transformers.disable_max_rows()

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "analysis" else Path.cwd()
RESULTS_ROOT = PROJECT_ROOT / "data" / "output"

# Maps each sector's case-folder prefix to a human-readable label used
# throughout this notebook (dropdowns, tooltips, chart titles).
SECTOR_CASE_PREFIXES = {
    "Aviation Cargo": "baseline-cargo-transition",
    "Maritime Cargo": "baseline-maritime-cargo-transition",
    "Aviation Passenger": "baseline-passenger-transition",
}

# The 8 policy/demand variants every sector was run for.
SCENARIO_VARIANTS = [
    "aktuellepolitiken-risk-neutral",
    "fokush2-risk-neutral",
    "fokusstrom-risk-neutral",
    "hohenachfrage-risk-neutral",
    "nachfrageniedrig-risk-neutral",
    "technologiemix-risk-neutral",
    "worstcasemean-risk-averse",
    "worstcasetailrisk-risk-averse",
]

# Paper naming convention: map each functional case id to its figure/table label.
# (Keys are the raw scenario_id variants used for file paths -- unchanged.)
CASE_LABEL = {
    "aktuellepolitiken-risk-neutral": "RN–ExPol",
    "fokush2-risk-neutral": "RN–H₂",
    "fokusstrom-risk-neutral": "RN–Elec",
    "hohenachfrage-risk-neutral": "RN–HighDem",
    "nachfrageniedrig-risk-neutral": "RN–LowDem",
    "technologiemix-risk-neutral": "RN–Mix",
    "worstcasemean-risk-averse": "Worst-case mean",
    "worstcasetailrisk-risk-averse": "Worst-case ES",
}
def case_label(variant: str) -> str:
    """Functional case id -> paper label (identity for anything unmapped)."""
    return CASE_LABEL.get(variant, variant)

found = 0
for sector_group, prefix in SECTOR_CASE_PREFIXES.items():
    for variant in SCENARIO_VARIANTS:
        if (RESULTS_ROOT / f"{prefix}-{variant}").exists():
            found += 1
print(f"Found {found}/{len(SECTOR_CASE_PREFIXES) * len(SCENARIO_VARIANTS)} sector x case combinations.")

import vl_convert as vlc
from IPython.display import display as _display


def show(chart: alt.Chart) -> None:
    """Renders an Altair chart via a multi-mimetype display bundle:

    - application/vnd.vegalite.v5+json: VS Code's and JupyterLab's Jupyter
      renderers draw this natively with their own bundled Vega-Lite --
      no JavaScript execution required, so it works even where a notebook
      frontend sandboxes/strips <script> tags from raw HTML output (the
      "cell runs, no error, but no figure appears" symptom).
    - text/html: a fully self-contained fallback (JS bundled inline via
      vl-convert, no CDN fetch) for frontends without native Vega-Lite
      mimetype support.

    A frontend picks whichever mimetype it understands; unsupported ones
    are ignored, so both can safely be provided together.
    """
    spec = chart.to_dict()
    html = vlc.vegalite_to_html(spec, bundle=True)
    _display(
        {
            "application/vnd.vegalite.v5+json": spec,
            "text/html": html,
            "text/plain": "<Vega-Lite chart>",
        },
        raw=True,
    )


Found 24/24 sector x case combinations.


## Load results

Loads all three sectors x all 8 scenario variants (24 cases) into combined
dataframes, each tagged with `sector_group` and `scenario` columns.
`aircraft_count`/`vessel_count` are unified into a single `stock_count`
column so charts don't need to know which sector they're looking at -- the
Sector and Case dropdowns on each chart just filter `sector_group` and
`scenario`. `aircraft.csv` (per-asset-per-year detail) is large across 24
cases, so only the columns the port/hub chart needs are read.

In [2]:
def _read_csv_if_exists(path: Path, usecols: list[str] | None = None) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    if usecols is None:
        return pd.read_csv(path, low_memory=False)
    header = pd.read_csv(path, nrows=0).columns
    return pd.read_csv(path, low_memory=False, usecols=[c for c in usecols if c in header])


AIRCRAFT_USECOLS = [
    "year",
    "main_hub_base",
    "main_hub",
    "primary_energy_carrier",
    "primary_energy_consumption",
    "secondary_energy_consumption",
]

technology_frames: list[pd.DataFrame] = []
energy_frames: list[pd.DataFrame] = []
aircraft_frames: list[pd.DataFrame] = []
investments_frames: list[pd.DataFrame] = []
selected_decisions_frames: list[pd.DataFrame] = []
summary_frames: list[pd.DataFrame] = []

for sector_group, prefix in SECTOR_CASE_PREFIXES.items():
    for variant in SCENARIO_VARIANTS:
        case_dir = RESULTS_ROOT / f"{prefix}-{variant}"
        if not case_dir.exists():
            continue

        this_is_aviation = (case_dir / "aviation_technology.csv").exists()
        this_is_maritime = (case_dir / "maritime_technology.csv").exists()
        if not (this_is_aviation or this_is_maritime):
            continue
        this_sector_prefix = "aviation" if this_is_aviation else "maritime"
        this_stock_col = "aircraft_count" if this_is_aviation else "vessel_count"

        tech = _read_csv_if_exists(case_dir / f"{this_sector_prefix}_technology.csv")
        if not tech.empty:
            tech = tech.rename(columns={this_stock_col: "stock_count"})
            tech["sector_group"] = sector_group
            tech["scenario"] = case_label(variant)
            technology_frames.append(tech)

        nrg = _read_csv_if_exists(case_dir / f"{this_sector_prefix}_energy_emissions.csv")
        if not nrg.empty:
            nrg = nrg.copy()
            nrg["sector_group"] = sector_group
            nrg["scenario"] = case_label(variant)
            energy_frames.append(nrg)

        ac = _read_csv_if_exists(case_dir / "aircraft.csv", usecols=AIRCRAFT_USECOLS)
        if not ac.empty:
            ac["sector_group"] = sector_group
            ac["scenario"] = case_label(variant)
            aircraft_frames.append(ac)

        inv = _read_csv_if_exists(case_dir / f"{this_sector_prefix}_investments.csv")
        if not inv.empty:
            inv = inv.rename(columns={this_stock_col: "stock_count"})
            inv["sector_group"] = sector_group
            inv["scenario"] = case_label(variant)
            investments_frames.append(inv)

        dec = _read_csv_if_exists(case_dir / "selected_ambiguity_aware_decisions.csv")
        if not dec.empty:
            dec = dec.copy()
            dec["sector_group"] = sector_group
            dec["scenario"] = case_label(variant)
            selected_decisions_frames.append(dec)

        raw_summary = _read_csv_if_exists(case_dir / "model_summary.csv")
        if not raw_summary.empty:
            s = raw_summary[["year"]].copy()
            s["alternative_share"] = raw_summary[f"{this_sector_prefix}_alternative_share"]
            s["transition_pressure"] = raw_summary[f"{this_sector_prefix}_transition_pressure"]
            s["effective_conventional_cost"] = raw_summary[
                f"{this_sector_prefix}_effective_conventional_cost"
            ]
            s["effective_alternative_cost"] = raw_summary[f"{this_sector_prefix}_effective_alternative_cost"]
            s["sector_group"] = sector_group
            s["scenario"] = case_label(variant)
            summary_frames.append(s)

technology = pd.concat(technology_frames, ignore_index=True) if technology_frames else pd.DataFrame()
energy = pd.concat(energy_frames, ignore_index=True) if energy_frames else pd.DataFrame()
aircraft = pd.concat(aircraft_frames, ignore_index=True) if aircraft_frames else pd.DataFrame()
investments_by_year = (
    pd.concat(investments_frames, ignore_index=True) if investments_frames else pd.DataFrame()
)
selected_decisions = (
    pd.concat(selected_decisions_frames, ignore_index=True) if selected_decisions_frames else pd.DataFrame()
)
summary = pd.concat(summary_frames, ignore_index=True) if summary_frames else pd.DataFrame()

assert not technology.empty, "No sector/case data found under data/output/"

sector_options = sorted(technology["sector_group"].unique().tolist())
case_options = sorted(technology["scenario"].unique().tolist())
stock_label = "Assets"

print(f"Sectors loaded: {sector_options}")
print(f"Cases loaded: {case_options}")
print(f"Technology rows: {len(technology)}, Energy rows: {len(energy)}, Aircraft rows: {len(aircraft)}")
technology.head()

Sectors loaded: ['Aviation Cargo', 'Aviation Passenger', 'Maritime Cargo']
Cases loaded: ['aktuellepolitiken-risk-neutral', 'fokush2-risk-neutral', 'fokusstrom-risk-neutral', 'hohenachfrage-risk-neutral', 'nachfrageniedrig-risk-neutral', 'technologiemix-risk-neutral', 'worstcasemean-risk-averse', 'worstcasetailrisk-risk-averse']
Technology rows: 30694, Energy rows: 30694, Aircraft rows: 508066


,year,application_name,operator_name,operator_country,segment,current_technology,primary_energy_carrier,secondary_energy_carrier,saf_pathway,stock_count,primary_energy_consumption,secondary_energy_consumption,total_emission,chargeable_emission,sector_group,scenario
0,2025,cargo,AeroLogic,Germany,long,kerosene_freight_long,kerosene,none,none,28,8.986177e+09,0.0,2.368756e+09,2.368756e+09,Aviation Cargo,aktuellepolitiken-risk-neutral
1,2025,cargo,DHL,Germany,long,kerosene_freight_long,kerosene,none,none,2,4.928886e+08,0.0,1.299254e+08,1.299254e+08,Aviation Cargo,aktuellepolitiken-risk-neutral
2,2025,cargo,DHL,Germany,long,kerosene_freight_medium,kerosene,none,none,6,2.270750e+08,0.0,5.985697e+07,5.985697e+07,Aviation Cargo,aktuellepolitiken-risk-neutral
3,2025,cargo,DHL,Germany,medium,kerosene_freight_medium,kerosene,none,none,26,9.547414e+08,0.0,2.516698e+08,2.516698e+08,Aviation Cargo,aktuellepolitiken-risk-neutral
4,2025,cargo,Lufthansa Cargo,Germany,long,kerosene_freight_long,kerosene,none,none,12,1.870256e+09,0.0,4.929996e+08,4.929996e+08,Aviation Cargo,aktuellepolitiken-risk-neutral


## Technology diffusion over time

Stacked area chart of fleet composition by technology, year over year. Use
the **Sector** and **Case** dropdowns to switch between the 3 sectors and 8
policy/demand variants. Click a legend entry to isolate one technology
(shift-click to add more); use the segment dropdown to filter by
short/medium/long or regional/deepsea.

In [3]:
sector_dropdown = alt.binding_select(options=sector_options, name="Sector: ")
sector_select = alt.selection_point(
    fields=["sector_group"],
    bind=sector_dropdown,
    value=[{"sector_group": "Aviation Passenger"}],
)

case_dropdown = alt.binding_select(options=case_options, name="Case: ")
case_select = alt.selection_point(
    fields=["scenario"],
    bind=case_dropdown,
    value=[{"scenario": "RN–ExPol"}],
)

segment_options = ["All"] + sorted(technology["segment"].dropna().astype(str).unique().tolist())
segment_dropdown = alt.binding_select(options=segment_options, name="Segment: ")
segment_select = alt.selection_point(
    fields=["segment_filter"],
    bind=segment_dropdown,
    value=[{"segment_filter": "All"}],
)

technology_plot = technology.copy()
technology_plot["segment_filter"] = technology_plot["segment"]

tech_by_year = (
    technology_plot.groupby(
        ["year", "sector_group", "scenario", "segment_filter", "current_technology"], as_index=False,
    )["stock_count"]
    .sum()
)

# Duplicate rows tagged "All" so the dropdown's "All" option sums every segment.
all_segments = (
    tech_by_year.groupby(
        ["year", "sector_group", "scenario", "current_technology"], as_index=False,
    )["stock_count"]
    .sum()
    .assign(segment_filter="All")
)
tech_by_year = pd.concat([tech_by_year, all_segments], ignore_index=True)

technology_legend_select = alt.selection_point(fields=["current_technology"], bind="legend")

diffusion_chart = (
    alt.Chart(tech_by_year)
    .mark_area()
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("stock_count:Q", stack="normalize", title="Share of fleet", axis=alt.Axis(format="%")),
        color=alt.Color("current_technology:N", title="Technology", scale=alt.Scale(scheme="tableau20")),
        opacity=alt.condition(technology_legend_select, alt.value(0.9), alt.value(0.08)),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("scenario:N", title="Case"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("current_technology:N", title="Technology"),
            alt.Tooltip("stock_count:Q", title=stock_label, format=",.0f"),
        ],
    )
    .add_params(sector_select, case_select, segment_select, technology_legend_select)
    .transform_filter(sector_select)
    .transform_filter(case_select)
    .transform_filter(segment_select)
    .properties(width=760, height=380, title="Technology diffusion over time (share of fleet)")
    .interactive()
)
show(diffusion_chart)

<Vega-Lite chart>

## Technology adoption per year

The diffusion chart above shows fleet *composition* (share of the total
fleet) at each year -- a stock view. This shows the *flow*: how many new
aircraft/vessels were actually invested in with each technology, in each
individual year, from `{sector}_investments.csv`. A technology can dominate
the stock view for years after its adoption flow has already dropped to
zero (fleet turnover is slow), so the two views tell different parts of the
same story. Same Sector and Case dropdowns as above.

In [4]:
if investments_by_year.empty:
    print("No investments CSV found for any sector.")
else:
    adoption_sector_dropdown = alt.binding_select(options=sector_options, name="Sector: ")
    adoption_sector_select = alt.selection_point(
        fields=["sector_group"],
        bind=adoption_sector_dropdown,
        value=[{"sector_group": "Aviation Passenger"}],
    )
    adoption_case_dropdown = alt.binding_select(options=case_options, name="Case: ")
    adoption_case_select = alt.selection_point(
        fields=["scenario"],
        bind=adoption_case_dropdown,
        value=[{"scenario": "RN–ExPol"}],
    )

    adoption_by_year = investments_by_year.groupby(
        ["year", "sector_group", "scenario", "current_technology"], as_index=False,
    )["stock_count"].sum()

    adoption_chart = (
        alt.Chart(adoption_by_year)
        .mark_bar()
        .encode(
            x=alt.X("year:O", title="Year"),
            y=alt.Y("stock_count:Q", title=f"New {stock_label.lower()} adopting technology"),
            color=alt.Color(
                "current_technology:N", title="Technology", scale=alt.Scale(scheme="tableau20"),
            ),
            tooltip=[
                alt.Tooltip("sector_group:N", title="Sector"),
                alt.Tooltip("scenario:N", title="Case"),
                alt.Tooltip("year:O", title="Year"),
                alt.Tooltip("current_technology:N", title="Technology"),
                alt.Tooltip("stock_count:Q", title=stock_label, format=",.0f"),
            ],
        )
        .add_params(adoption_sector_select, adoption_case_select)
        .transform_filter(adoption_sector_select)
        .transform_filter(adoption_case_select)
        .properties(
            width=760,
            height=380,
            title=f"Technology adoption per year (new {stock_label.lower()} by technology)",
        )
    )
    show(adoption_chart)

<Vega-Lite chart>

## Energy demand by carrier over time

Primary + secondary energy consumption per carrier, year over year, in the
model's native `kWh`. Sector and Case dropdowns plus the same segment
filter as above (independent dropdowns rather than a shared cross-chart
selection, since that requires Altair's newer cross-chart param wiring;
this notebook keeps each chart's filters independent so it works reliably
across Altair versions).

In [5]:
energy_plot = energy.copy()
energy_plot["segment_filter"] = energy_plot["segment"]

primary = (
    energy_plot.groupby(
        ["year", "sector_group", "scenario", "segment_filter", "primary_energy_carrier"], as_index=False,
    )["primary_energy_consumption"]
    .sum()
    .rename(columns={"primary_energy_carrier": "carrier", "primary_energy_consumption": "energy_kwh"})
)
secondary_mask = energy_plot["secondary_energy_carrier"].fillna("").astype(str).str.strip().ne("")
secondary_mask &= energy_plot["secondary_energy_carrier"].astype(str).str.lower().ne("none")
secondary = (
    energy_plot.loc[secondary_mask]
    .groupby(
        ["year", "sector_group", "scenario", "segment_filter", "secondary_energy_carrier"], as_index=False,
    )["secondary_energy_consumption"]
    .sum()
    .rename(
        columns={
            "secondary_energy_carrier": "carrier",
            "secondary_energy_consumption": "energy_kwh",
        },
    )
)
energy_by_carrier = pd.concat([primary, secondary], ignore_index=True)
energy_by_carrier = energy_by_carrier.groupby(
    ["year", "sector_group", "scenario", "segment_filter", "carrier"], as_index=False,
)["energy_kwh"].sum()

all_segments_energy = (
    energy_by_carrier.groupby(
        ["year", "sector_group", "scenario", "carrier"], as_index=False,
    )["energy_kwh"]
    .sum()
    .assign(segment_filter="All")
)
energy_by_carrier = pd.concat([energy_by_carrier, all_segments_energy], ignore_index=True)
energy_by_carrier["energy_twh"] = energy_by_carrier["energy_kwh"] / 1_000_000_000.0

energy_sector_dropdown = alt.binding_select(options=sector_options, name="Sector: ")
energy_sector_select = alt.selection_point(
    fields=["sector_group"],
    bind=energy_sector_dropdown,
    value=[{"sector_group": "Aviation Passenger"}],
)
energy_case_dropdown = alt.binding_select(options=case_options, name="Case: ")
energy_case_select = alt.selection_point(
    fields=["scenario"],
    bind=energy_case_dropdown,
    value=[{"scenario": "RN–ExPol"}],
)
energy_segment_dropdown = alt.binding_select(options=segment_options, name="Segment: ")
energy_segment_select = alt.selection_point(
    fields=["segment_filter"],
    bind=energy_segment_dropdown,
    value=[{"segment_filter": "All"}],
)
carrier_legend_select = alt.selection_point(fields=["carrier"], bind="legend")

energy_chart = (
    alt.Chart(energy_by_carrier)
    .mark_area()
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("energy_twh:Q", title="Energy demand (TWh)", stack="zero"),
        color=alt.Color("carrier:N", title="Energy carrier", scale=alt.Scale(scheme="set2")),
        opacity=alt.condition(carrier_legend_select, alt.value(0.9), alt.value(0.08)),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("scenario:N", title="Case"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("carrier:N", title="Carrier"),
            alt.Tooltip("energy_twh:Q", title="Energy (TWh)", format=",.3f"),
        ],
    )
    .add_params(energy_sector_select, energy_case_select, energy_segment_select, carrier_legend_select)
    .transform_filter(energy_sector_select)
    .transform_filter(energy_case_select)
    .transform_filter(energy_segment_select)
    .properties(width=760, height=380, title="Energy demand by carrier over time")
    .interactive()
)
show(energy_chart)

<Vega-Lite chart>

## Energy demand by port / hub

Real per-asset hub/port data (`main_hub_base`, asset-level and
activity-profile-derived for aviation; falls back to `main_hub` -- the home
port for maritime) from `aircraft.csv`, broken down by energy carrier. Use
the Sector and Case dropdowns and the year slider to see how demand shifts
between hubs/ports and carriers over the simulation horizon.

In [6]:
assert not aircraft.empty, "No aircraft.csv found for any sector"

hub_column = "main_hub_base" if "main_hub_base" in aircraft.columns else "main_hub"
port_energy = aircraft.copy()
port_energy["hub"] = port_energy[hub_column].fillna("").astype(str).str.strip()
port_energy = port_energy.loc[port_energy["hub"].ne("")]
port_energy["carrier"] = port_energy["primary_energy_carrier"].fillna("").astype(str).str.lower()
port_energy["energy_kwh"] = pd.to_numeric(
    port_energy.get("primary_energy_consumption", 0.0), errors="coerce",
).fillna(0.0) + pd.to_numeric(
    port_energy.get("secondary_energy_consumption", 0.0), errors="coerce",
).fillna(0.0)

port_energy_by_year = (
    port_energy.groupby(
        ["year", "sector_group", "scenario", "hub", "carrier"], as_index=False,
    )["energy_kwh"]
    .sum()
)
port_energy_by_year["energy_gwh"] = port_energy_by_year["energy_kwh"] / 1_000_000.0

port_sector_dropdown = alt.binding_select(options=sector_options, name="Sector: ")
port_sector_select = alt.selection_point(
    fields=["sector_group"],
    bind=port_sector_dropdown,
    value=[{"sector_group": "Aviation Passenger"}],
)
port_case_dropdown = alt.binding_select(options=case_options, name="Case: ")
port_case_select = alt.selection_point(
    fields=["scenario"],
    bind=port_case_dropdown,
    value=[{"scenario": "RN–ExPol"}],
)

port_years = sorted(port_energy_by_year["year"].dropna().astype(int).unique().tolist())
port_year_select = alt.selection_point(
    fields=["year"],
    bind=alt.binding_range(min=port_years[0], max=port_years[-1], step=1, name="Year: "),
    value=[{"year": port_years[-1]}],
)

port_chart = (
    alt.Chart(port_energy_by_year)
    .mark_bar()
    .encode(
        x=alt.X("energy_gwh:Q", title="Energy demand (GWh)"),
        y=alt.Y("hub:N", title="Hub / port", sort="-x"),
        color=alt.Color("carrier:N", title="Energy carrier", scale=alt.Scale(scheme="set2")),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("scenario:N", title="Case"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("hub:N", title="Hub / port"),
            alt.Tooltip("carrier:N", title="Carrier"),
            alt.Tooltip("energy_gwh:Q", title="Energy (GWh)", format=",.2f"),
        ],
    )
    .add_params(port_sector_select, port_case_select, port_year_select)
    .transform_filter(port_sector_select)
    .transform_filter(port_case_select)
    .transform_filter(port_year_select)
    .properties(
        width=760,
        height=max(220, 24 * port_energy_by_year["hub"].nunique()),
        title="Energy demand by port / hub (drag the year slider)",
    )
)
show(port_chart)

<Vega-Lite chart>

## Fleet transition summary

Alternative-technology share and transition pressure across the run, for
the sector and case selected in the dropdowns.

In [7]:
transition_sector_dropdown = alt.binding_select(options=sector_options, name="Sector: ")
transition_sector_select = alt.selection_point(
    fields=["sector_group"],
    bind=transition_sector_dropdown,
    value=[{"sector_group": "Aviation Passenger"}],
)
transition_case_dropdown = alt.binding_select(options=case_options, name="Case: ")
transition_case_select = alt.selection_point(
    fields=["scenario"],
    bind=transition_case_dropdown,
    value=[{"scenario": "RN–ExPol"}],
)

transition_long = summary.melt(
    id_vars=["year", "sector_group", "scenario"],
    value_vars=["alternative_share", "transition_pressure"],
    var_name="metric",
    value_name="value",
)
transition_long["metric"] = transition_long["metric"].map(
    {"alternative_share": "Alternative share", "transition_pressure": "Transition pressure"},
)

transition_chart = (
    alt.Chart(transition_long)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("value:Q", title="Share", axis=alt.Axis(format="%")),
        color=alt.Color("metric:N", title="Metric"),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("scenario:N", title="Case"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("metric:N", title="Metric"),
            alt.Tooltip("value:Q", title="Value", format=".1%"),
        ],
    )
    .add_params(transition_sector_select, transition_case_select)
    .transform_filter(transition_sector_select)
    .transform_filter(transition_case_select)
    .properties(width=760, height=340, title="Alternative share & transition pressure over time")
    .interactive()
)
show(transition_chart)

<Vega-Lite chart>

## Operator-level fleet stock

Final-year fleet composition by operator and technology, for a quick sanity
check of who is flying/sailing what. Sector and Case dropdowns select which
sector/scenario's operators are shown.

In [8]:
operator_sector_dropdown = alt.binding_select(options=sector_options, name="Sector: ")
operator_sector_select = alt.selection_point(
    fields=["sector_group"],
    bind=operator_sector_dropdown,
    value=[{"sector_group": "Aviation Passenger"}],
)
operator_case_dropdown = alt.binding_select(options=case_options, name="Case: ")
operator_case_select = alt.selection_point(
    fields=["scenario"],
    bind=operator_case_dropdown,
    value=[{"scenario": "RN–ExPol"}],
)

final_year = int(technology["year"].max())
final_year_stock = (
    technology.loc[technology["year"] == final_year]
    .groupby(
        ["sector_group", "scenario", "operator_name", "current_technology"], as_index=False,
    )["stock_count"]
    .sum()
)

operator_chart = (
    alt.Chart(final_year_stock)
    .mark_bar()
    .encode(
        x=alt.X("stock_count:Q", title=stock_label),
        y=alt.Y("operator_name:N", title="Operator", sort="-x"),
        color=alt.Color("current_technology:N", title="Technology", scale=alt.Scale(scheme="tableau20")),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("scenario:N", title="Case"),
            alt.Tooltip("operator_name:N", title="Operator"),
            alt.Tooltip("current_technology:N", title="Technology"),
            alt.Tooltip("stock_count:Q", title=stock_label, format=",.0f"),
        ],
    )
    .add_params(operator_sector_select, operator_case_select)
    .transform_filter(operator_sector_select)
    .transform_filter(operator_case_select)
    .properties(
        width=760,
        height=max(220, 28 * final_year_stock["operator_name"].nunique()),
        title=f"Fleet by operator and technology ({final_year})",
    )
)
show(operator_chart)

<Vega-Lite chart>

## Financial analysis

Real financial outputs from the simulation's ambiguity-aware, NPV-based
decision framework: capex spending, the NPV of the technology actually
chosen at each replacement/growth decision, and how that expected NPV
compares against the framework's own worst-case and regret metrics (the
risk-aware side of the same decisions).

### Capex spending over time

Real investment cost (`investment_cost_eur`) by year and technology, from
`{sector}_investments.csv`. Sector and Case dropdowns as above.

In [9]:
if investments_by_year.empty:
    print("No investments CSV found for any sector.")
else:
    capex_sector_dropdown = alt.binding_select(options=sector_options, name="Sector: ")
    capex_sector_select = alt.selection_point(
        fields=["sector_group"],
        bind=capex_sector_dropdown,
        value=[{"sector_group": "Aviation Passenger"}],
    )
    capex_case_dropdown = alt.binding_select(options=case_options, name="Case: ")
    capex_case_select = alt.selection_point(
        fields=["scenario"],
        bind=capex_case_dropdown,
        value=[{"scenario": "RN–ExPol"}],
    )

    capex_by_year = investments_by_year.groupby(
        ["year", "sector_group", "scenario", "current_technology"], as_index=False,
    )["investment_cost_eur"].sum()
    capex_by_year["investment_cost_eur_m"] = capex_by_year["investment_cost_eur"] / 1_000_000.0

    capex_chart = (
        alt.Chart(capex_by_year)
        .mark_bar()
        .encode(
            x=alt.X("year:O", title="Year"),
            y=alt.Y("investment_cost_eur_m:Q", title="Investment (EUR millions)"),
            color=alt.Color("current_technology:N", title="Technology", scale=alt.Scale(scheme="tableau20")),
            tooltip=[
                alt.Tooltip("sector_group:N", title="Sector"),
                alt.Tooltip("scenario:N", title="Case"),
                alt.Tooltip("year:O", title="Year"),
                alt.Tooltip("current_technology:N", title="Technology"),
                alt.Tooltip("investment_cost_eur_m:Q", title="Investment (EUR M)", format=",.1f"),
            ],
        )
        .add_params(capex_sector_select, capex_case_select)
        .transform_filter(capex_sector_select)
        .transform_filter(capex_case_select)
        .properties(width=760, height=380, title="Capex spending over time")
    )
    show(capex_chart)

<Vega-Lite chart>

### NPV of selected decisions over time

`selected_ambiguity_aware_decisions.csv` records, for every replacement or
growth decision, the expected NPV of the technology actually chosen (as
opposed to `ambiguity_decision_scores.csv`, which has every *candidate*
technology considered, selected or not). Plotted here as the mean expected
NPV per year across all decisions made that year, for the sector/case
selected.

In [10]:
if selected_decisions.empty:
    print(
        "No selected_ambiguity_aware_decisions.csv found -- these runs may not use "
        "investment_logic=ambiguity_aware_utility."
    )
else:
    npv_sector_dropdown = alt.binding_select(options=sector_options, name="Sector: ")
    npv_sector_select = alt.selection_point(
        fields=["sector_group"],
        bind=npv_sector_dropdown,
        value=[{"sector_group": "Aviation Passenger"}],
    )
    npv_case_dropdown = alt.binding_select(options=case_options, name="Case: ")
    npv_case_select = alt.selection_point(
        fields=["scenario"],
        bind=npv_case_dropdown,
        value=[{"scenario": "RN–ExPol"}],
    )

    npv_by_year = (
        selected_decisions.groupby(
            ["selected_decision_year", "sector_group", "scenario"], as_index=False,
        )["selected_expected_npv"]
        .mean()
        .rename(columns={"selected_decision_year": "year"})
    )
    npv_by_year["expected_npv_m"] = npv_by_year["selected_expected_npv"] / 1_000_000.0

    npv_chart = (
        alt.Chart(npv_by_year)
        .mark_line(point=True)
        .encode(
            x=alt.X("year:O", title="Year"),
            y=alt.Y("expected_npv_m:Q", title="Mean expected NPV (EUR millions)"),
            tooltip=[
                alt.Tooltip("sector_group:N", title="Sector"),
                alt.Tooltip("scenario:N", title="Case"),
                alt.Tooltip("year:O", title="Year"),
                alt.Tooltip("expected_npv_m:Q", title="Mean expected NPV (EUR M)", format=",.1f"),
            ],
        )
        .add_params(npv_sector_select, npv_case_select)
        .transform_filter(npv_sector_select)
        .transform_filter(npv_case_select)
        .properties(
            width=760,
            height=340,
            title="Mean expected NPV of selected technology decisions over time",
        )
        .interactive()
    )
    show(npv_chart)

<Vega-Lite chart>

### Expected vs. risk-adjusted NPV

The same selected decisions, but comparing the plain expected NPV against
the ambiguity-aware framework's worst-case-mean NPV and max regret --
showing how much of the "expected" return the risk-averse view of the same
decision actually gives up. Sector and Case dropdowns as above.

In [11]:
if selected_decisions.empty:
    print("No selected_ambiguity_aware_decisions.csv found; skipping.")
else:
    risk_sector_dropdown = alt.binding_select(options=sector_options, name="Sector: ")
    risk_sector_select = alt.selection_point(
        fields=["sector_group"],
        bind=risk_sector_dropdown,
        value=[{"sector_group": "Aviation Passenger"}],
    )
    risk_case_dropdown = alt.binding_select(options=case_options, name="Case: ")
    risk_case_select = alt.selection_point(
        fields=["scenario"],
        bind=risk_case_dropdown,
        value=[{"scenario": "RN–ExPol"}],
    )

    risk_columns = {
        "selected_expected_npv": "Expected NPV",
        "selected_robust_worst_case_mean_npv": "Worst-case-mean NPV",
        "selected_max_regret": "Max regret",
    }
    risk_by_year = (
        selected_decisions.groupby(
            ["selected_decision_year", "sector_group", "scenario"], as_index=False,
        )[list(risk_columns)]
        .mean()
        .rename(columns={"selected_decision_year": "year"})
    )
    risk_long = risk_by_year.melt(
        id_vars=["year", "sector_group", "scenario"], var_name="metric", value_name="value_eur",
    )
    risk_long["metric"] = risk_long["metric"].map(risk_columns)
    risk_long["value_eur_m"] = risk_long["value_eur"] / 1_000_000.0

    risk_chart = (
        alt.Chart(risk_long)
        .mark_line(point=True)
        .encode(
            x=alt.X("year:O", title="Year"),
            y=alt.Y("value_eur_m:Q", title="EUR millions"),
            color=alt.Color("metric:N", title="Metric"),
            tooltip=[
                alt.Tooltip("sector_group:N", title="Sector"),
                alt.Tooltip("scenario:N", title="Case"),
                alt.Tooltip("year:O", title="Year"),
                alt.Tooltip("metric:N", title="Metric"),
                alt.Tooltip("value_eur_m:Q", title="EUR M", format=",.1f"),
            ],
        )
        .add_params(risk_sector_select, risk_case_select)
        .transform_filter(risk_sector_select)
        .transform_filter(risk_case_select)
        .properties(
            width=760,
            height=380,
            title="Expected vs. risk-adjusted NPV of selected decisions",
        )
        .interactive()
    )
    show(risk_chart)

<Vega-Lite chart>

### Effective operating cost: conventional vs. alternative

From `model_summary.csv` -- the crossover point where the alternative
technology's effective operating cost drops below the conventional
technology's is the point at which the transition becomes economically
self-sustaining rather than policy/mandate-driven. Sector and Case
dropdowns as above.

In [12]:
cost_sector_dropdown = alt.binding_select(options=sector_options, name="Sector: ")
cost_sector_select = alt.selection_point(
    fields=["sector_group"],
    bind=cost_sector_dropdown,
    value=[{"sector_group": "Aviation Passenger"}],
)
cost_case_dropdown = alt.binding_select(options=case_options, name="Case: ")
cost_case_select = alt.selection_point(
    fields=["scenario"],
    bind=cost_case_dropdown,
    value=[{"scenario": "RN–ExPol"}],
)

cost_long = summary.melt(
    id_vars=["year", "sector_group", "scenario"],
    value_vars=["effective_conventional_cost", "effective_alternative_cost"],
    var_name="technology_path",
    value_name="cost_eur",
)
cost_long["technology_path"] = cost_long["technology_path"].map(
    {"effective_conventional_cost": "Conventional", "effective_alternative_cost": "Alternative"},
)
cost_long["cost_eur_m"] = cost_long["cost_eur"] / 1_000_000.0

cost_chart = (
    alt.Chart(cost_long)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("cost_eur_m:Q", title="Effective operating cost (EUR millions)"),
        color=alt.Color(
            "technology_path:N",
            title="Technology path",
            scale=alt.Scale(domain=["Conventional", "Alternative"], range=["#7f7f7f", "#2a9d8f"]),
        ),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("scenario:N", title="Case"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("technology_path:N", title="Path"),
            alt.Tooltip("cost_eur_m:Q", title="Cost (EUR M)", format=",.1f"),
        ],
    )
    .add_params(cost_sector_select, cost_case_select)
    .transform_filter(cost_sector_select)
    .transform_filter(cost_case_select)
    .properties(
        width=760,
        height=340,
        title="Effective operating cost: conventional vs. alternative technology",
    )
    .interactive()
)
show(cost_chart)

<Vega-Lite chart>

## Hourly energy timeseries

The full hourly-resolved dataset (per aircraft/vessel, year, energy type,
region) lives in `hourly_energy_timeseries.parquet` inside each case's
results folder, generated separately with:

```bash
natm-hourly-timeseries --results data/output/<run-name>
```

It disaggregates each asset's real annual energy total using a real
hour-of-week activity shape derived from historical OpenSky flight
timestamps (aviation) or Global Fishing Watch AIS port-visit timestamps
(maritime) -- summing any asset's hourly values back up reproduces its
annual total exactly. These files are large (tens of millions of rows per
sector/run), so this notebook only loads small, pre-aggregated slices via
`pyarrow.parquet` filters/columns rather than reading them whole, for
whichever sector x case combinations have one generated.

In [13]:
import pyarrow.parquet as pq

hourly_frames: list[pd.DataFrame] = []
latest_year = int(technology["year"].max())
missing_hourly: list[str] = []

for sector_group, prefix in SECTOR_CASE_PREFIXES.items():
    for variant in SCENARIO_VARIANTS:
        hourly_path = RESULTS_ROOT / f"{prefix}-{variant}" / "hourly_energy_timeseries.parquet"
        if not hourly_path.exists():
            missing_hourly.append(f"{sector_group} / {variant}")
            continue

        pf = pq.ParquetFile(hourly_path)
        hourly_year = pd.read_parquet(
            hourly_path,
            columns=["timestamp", "energy_type", "energy_kwh"],
            filters=[("year", "=", latest_year)],
        )
        hourly_year["sector_group"] = sector_group
        hourly_year["scenario"] = case_label(variant)
        hourly_frames.append(hourly_year)

if missing_hourly:
    print(f"No hourly timeseries generated for {len(missing_hourly)} sector x case combinations.")
    print("Generate one with: natm-hourly-timeseries --results data/output/<run-name>")

if not hourly_frames:
    print("\nNo hourly timeseries found for any sector/case combination.")
else:
    hourly_all = pd.concat(hourly_frames, ignore_index=True)
    hourly_all["hour_of_day"] = hourly_all["timestamp"].dt.hour
    profile = hourly_all.groupby(
        ["hour_of_day", "sector_group", "scenario", "energy_type"], as_index=False,
    )["energy_kwh"].mean()

    hourly_sector_options = sorted(profile["sector_group"].unique().tolist())
    hourly_case_options = sorted(profile["scenario"].unique().tolist())

    hourly_sector_dropdown = alt.binding_select(options=hourly_sector_options, name="Sector: ")
    hourly_sector_select = alt.selection_point(
        fields=["sector_group"],
        bind=hourly_sector_dropdown,
        value=[{"sector_group": hourly_sector_options[0]}],
    )
    hourly_case_dropdown = alt.binding_select(options=hourly_case_options, name="Case: ")
    hourly_case_select = alt.selection_point(
        fields=["scenario"],
        bind=hourly_case_dropdown,
        value=[{"scenario": hourly_case_options[0]}],
    )

    profile_chart = (
        alt.Chart(profile)
        .mark_line(point=True)
        .encode(
            x=alt.X("hour_of_day:O", title="Hour of day (UTC)"),
            y=alt.Y("energy_kwh:Q", title="Mean hourly energy demand (kWh)"),
            color=alt.Color("energy_type:N", title="Energy type", scale=alt.Scale(scheme="set2")),
            tooltip=[
                alt.Tooltip("sector_group:N", title="Sector"),
                alt.Tooltip("scenario:N", title="Case"),
                alt.Tooltip("hour_of_day:O", title="Hour (UTC)"),
                alt.Tooltip("energy_type:N", title="Energy type"),
                alt.Tooltip("energy_kwh:Q", title="Mean demand (kWh)", format=",.1f"),
            ],
        )
        .add_params(hourly_sector_select, hourly_case_select)
        .transform_filter(hourly_sector_select)
        .transform_filter(hourly_case_select)
        .properties(
            width=760,
            height=340,
            title=f"Real-shape average hourly demand profile ({latest_year})",
        )
    )
    show(profile_chart)

No hourly timeseries generated for 24 sector x case combinations.
Generate one with: natm-hourly-timeseries --results data/output/<run-name>

No hourly timeseries found for any sector/case combination.


# Compare all cases (3 sectors x 8 policy/demand variants)

Every sector -- aviation cargo, maritime cargo, aviation passenger -- was run
for the same 8 policy/demand scenario variants (`aktuellepolitiken`,
`fokush2`, `fokusstrom`, `hohenachfrage`, `nachfrageniedrig`,
`technologiemix`, `worstcasemean-risk-averse`, `worstcasetailrisk-risk-averse`),
24 cases total.

This section auto-discovers all 24 `data/output/baseline-<sector>-transition-<variant>`
folders across all three sectors and plots every chart faceted by sector
(one small-multiple panel per sector, independent scales since absolute
fleet size/energy/cost differ a lot across sectors) and colored by scenario
-- so every case and every sector is visible together in one view, with no
variable to edit and no cell to re-run per sector.

In [14]:
def _scenario_label(prefix: str, path: Path) -> str:
    return case_label(path.name[len(prefix) + 1 :])


compare_summaries: list[pd.DataFrame] = []
compare_energy: list[pd.DataFrame] = []
compare_investments: list[pd.DataFrame] = []

for group_label, prefix in SECTOR_CASE_PREFIXES.items():
    case_dirs = sorted(p for p in RESULTS_ROOT.glob(f"{prefix}-*") if p.is_dir())
    if not case_dirs:
        print(f"{group_label}: no scenario variants found for '{prefix}-*'")
        continue
    print(f"{group_label}: {len(case_dirs)} scenario variants")

    for path in case_dirs:
        label = _scenario_label(prefix, path)
        this_is_aviation = (path / "aviation_technology.csv").exists()
        this_is_maritime = (path / "maritime_technology.csv").exists()
        if not (this_is_aviation or this_is_maritime):
            print(f"  Skipping {label}: no aviation_technology.csv or maritime_technology.csv found.")
            continue
        this_sector_prefix = "aviation" if this_is_aviation else "maritime"

        raw_summary = _read_csv_if_exists(path / "model_summary.csv")
        if not raw_summary.empty:
            scenario_summary = raw_summary[["year"]].copy()
            scenario_summary["alternative_share"] = raw_summary[f"{this_sector_prefix}_alternative_share"]
            scenario_summary["transition_pressure"] = raw_summary[
                f"{this_sector_prefix}_transition_pressure"
            ]
            scenario_summary["sector_group"] = group_label
            scenario_summary["scenario"] = label
            compare_summaries.append(scenario_summary)

        scenario_energy = _read_csv_if_exists(path / f"{this_sector_prefix}_energy_emissions.csv")
        if not scenario_energy.empty:
            scenario_energy = scenario_energy.copy()
            scenario_energy["sector_group"] = group_label
            scenario_energy["scenario"] = label
            compare_energy.append(scenario_energy)

        scenario_investments = _read_csv_if_exists(path / f"{this_sector_prefix}_investments.csv")
        if not scenario_investments.empty:
            scenario_investments = scenario_investments.copy()
            scenario_investments["sector_group"] = group_label
            scenario_investments["scenario"] = label
            compare_investments.append(scenario_investments)

compare_summary = pd.concat(compare_summaries, ignore_index=True) if compare_summaries else pd.DataFrame()
compare_energy_df = pd.concat(compare_energy, ignore_index=True) if compare_energy else pd.DataFrame()
compare_investments_df = (
    pd.concat(compare_investments, ignore_index=True) if compare_investments else pd.DataFrame()
)

n_cases = (
    compare_summary[["sector_group", "scenario"]].drop_duplicates().shape[0]
    if not compare_summary.empty
    else 0
)
print(f"\nTotal cases loaded: {n_cases}")
print(f"Combined summary rows: {len(compare_summary)}, energy rows: {len(compare_energy_df)}")

Aviation Cargo: 8 scenario variants
Maritime Cargo: 8 scenario variants
Aviation Passenger: 8 scenario variants

Total cases loaded: 24
Combined summary rows: 624, energy rows: 30694


## Alternative technology share over time, all sectors and scenarios

In [15]:
share_compare_chart = (
    alt.Chart(compare_summary)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("alternative_share:Q", title="Alternative technology share", axis=alt.Axis(format="%")),
        color=alt.Color("scenario:N", title="Scenario", scale=alt.Scale(scheme="tableau20")),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("scenario:N", title="Scenario"),
            alt.Tooltip("alternative_share:Q", title="Alternative share", format=".1%"),
        ],
    )
    .properties(width=230, height=280)
    .facet(
        column=alt.Column("sector_group:N", title=None),
        title="Alternative technology share by scenario, all sectors",
    )
    .resolve_scale(y="independent")
)
show(share_compare_chart)

<Vega-Lite chart>

## Transition pressure over time, all sectors and scenarios

In [16]:
pressure_compare_chart = (
    alt.Chart(compare_summary)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("transition_pressure:Q", title="Transition pressure", axis=alt.Axis(format="%")),
        color=alt.Color("scenario:N", title="Scenario", scale=alt.Scale(scheme="tableau20")),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("scenario:N", title="Scenario"),
            alt.Tooltip("transition_pressure:Q", title="Transition pressure", format=".1%"),
        ],
    )
    .properties(width=230, height=280)
    .facet(
        column=alt.Column("sector_group:N", title=None),
        title="Transition pressure by scenario, all sectors",
    )
    .resolve_scale(y="independent")
)
show(pressure_compare_chart)

<Vega-Lite chart>

## Total energy demand over time, all sectors and scenarios

Sum of primary + secondary energy consumption across the whole fleet, per
year, per scenario, faceted by sector (independent y-scales since absolute
fleet size differs a lot between sectors).

In [17]:
compare_energy_df["energy_kwh_total"] = pd.to_numeric(
    compare_energy_df.get("primary_energy_consumption", 0.0), errors="coerce",
).fillna(0.0) + pd.to_numeric(
    compare_energy_df.get("secondary_energy_consumption", 0.0), errors="coerce",
).fillna(0.0)

energy_totals_by_scenario = compare_energy_df.groupby(
    ["year", "sector_group", "scenario"], as_index=False,
)["energy_kwh_total"].sum()
energy_totals_by_scenario["energy_twh"] = energy_totals_by_scenario["energy_kwh_total"] / 1_000_000_000.0

# Click a scenario in the legend to isolate it (shift-click to add more);
# click empty space to reset. Faceted small multiples make 8 overlapping
# lines per panel hard to tell apart by color alone, so this lets you pick
# one scenario at a time instead of squinting at the legend swatches.
scenario_selection = alt.selection_point(fields=["scenario"], bind="legend")

energy_compare_chart = (
    alt.Chart(energy_totals_by_scenario)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("energy_twh:Q", title="Total energy demand (TWh)"),
        color=alt.Color("scenario:N", title="Scenario", scale=alt.Scale(scheme="tableau20")),
        opacity=alt.condition(scenario_selection, alt.value(1.0), alt.value(0.08)),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("scenario:N", title="Scenario"),
            alt.Tooltip("energy_twh:Q", title="Energy (TWh)", format=",.3f"),
        ],
    )
    .add_params(scenario_selection)
    .properties(width=230, height=280)
    .facet(
        column=alt.Column("sector_group:N", title=None),
        title="Total energy demand by scenario, all sectors (click a scenario in the legend to isolate it)",
    )
    .resolve_scale(y="independent")
)
show(energy_compare_chart)

<Vega-Lite chart>

## Electricity demand over time, all sectors and scenarios

Fuel energy alone does not say how much *electricity* a scenario needs --
that only exists for carriers actually made from power: e-kerosene (power-
to-liquid), renewable hydrogen (electrolysis), and any direct-electricity
carrier the technology catalog defines (`electricity`/`battery`; none of
the loaded cases actually adopt hybrid-electric aircraft, so those two are
present in the code but empty here). Everything else -- kerosene, SAF/HEFA
(bio-based, not power-derived), LNG, marine diesel, biofuel -- draws no
electricity and is excluded.

**The model itself has no conversion-efficiency parameter.** It records
fuel energy and emission factors, not the power needed upstream to make an
e-fuel, so this section is a documented assumption layered on top of the
run results, not something read out of them. The efficiencies below
default to conventional literature values and are Vega-Lite parameters
bound to range sliders on the chart itself -- drag them and the chart
recomputes with no cell re-run needed, in both the live kernel and any
static HTML export.

In [18]:
# Which modelled carriers are made from electricity, and which
# consumption column and (Sector x Case)-loaded frame column records their
# energy. Mirrors the same mapping used in the cartographic dashboard
# (analysis/cartographic_dashboard.py -- ELECTRICITY_SOURCES), duplicated
# here deliberately rather than imported: this notebook and the dashboard
# are meant to stand alone.
ELECTRICITY_SOURCE_MAP = {
    "e_kerosene": {
        "carrier_column": "secondary_energy_carrier",
        "field": "secondary_energy_consumption",
        "label": "E-kerosene (PtL)",
        "default_efficiency": 0.48,
    },
    "hydrogen": {
        "carrier_column": "primary_energy_carrier",
        "field": "primary_energy_consumption",
        "label": "Renewable hydrogen",
        "default_efficiency": 0.68,
    },
    "electricity": {
        "carrier_column": "primary_energy_carrier",
        "field": "primary_energy_consumption",
        "label": "Direct electricity",
        "default_efficiency": 0.95,
    },
    "battery": {
        "carrier_column": "secondary_energy_carrier",
        "field": "secondary_energy_consumption",
        "label": "Direct electricity (battery)",
        "default_efficiency": 0.95,
    },
    # Maritime hybrid propulsion (hybrid_regional/_deepsea) records its
    # battery-assist energy as secondary_energy_carrier == "electricity" --
    # the same carrier *string* as the "electricity" entry above, but that
    # one is scoped to primary_energy_carrier (a battery-electric vessel's
    # only fuel). match_value decouples the dict key (a unique identifier)
    # from the actual string compared against the column, so both can
    # target the literal value "electricity" without a key collision.
    "hybrid_electricity": {
        "carrier_column": "secondary_energy_carrier",
        "match_value": "electricity",
        "field": "secondary_energy_consumption",
        "label": "Hybrid propulsion electricity",
        "default_efficiency": 0.95,
    },
}

assert not compare_energy_df.empty, "Run the 'Compare all cases' section above first."

_electricity_fuel_rows = []
for _source, _spec in ELECTRICITY_SOURCE_MAP.items():
    _carrier_col, _field = _spec["carrier_column"], _spec["field"]
    _match_value = _spec.get("match_value", _source)
    _matched = compare_energy_df.loc[
        compare_energy_df[_carrier_col].astype(str).str.strip().str.lower().eq(_match_value)
    ]
    if _matched.empty:
        continue
    _totals = _matched.groupby(
        ["year", "sector_group", "scenario"], as_index=False,
    )[_field].sum().rename(columns={_field: "fuel_kwh"})
    _totals["source"] = _source
    _totals["source_label"] = _spec["label"]
    _electricity_fuel_rows.append(_totals)

electricity_fuel_df = (
    pd.concat(_electricity_fuel_rows, ignore_index=True)
    if _electricity_fuel_rows
    else pd.DataFrame(columns=["year", "sector_group", "scenario", "fuel_kwh", "source", "source_label"])
)

_sources_present = sorted(electricity_fuel_df["source"].unique().tolist())
print(f"Electricity-derived carriers present in the loaded cases: {_sources_present or 'none'}")
if "e_kerosene" not in _sources_present:
    print("(no e-kerosene rows -- check the SAF mandate actually triggers PtL adoption in these cases)")
electricity_fuel_df.head()

Electricity-derived carriers present in the loaded cases: ['e_kerosene', 'hybrid_electricity', 'hydrogen']


,year,sector_group,scenario,fuel_kwh,source,source_label
0,2028,Aviation Passenger,aktuellepolitiken-risk-neutral,8.044613e+07,e_kerosene,E-kerosene (PtL)
1,2028,Aviation Passenger,fokush2-risk-neutral,8.044613e+07,e_kerosene,E-kerosene (PtL)
2,2028,Aviation Passenger,fokusstrom-risk-neutral,8.044613e+07,e_kerosene,E-kerosene (PtL)
3,2028,Aviation Passenger,hohenachfrage-risk-neutral,8.044613e+07,e_kerosene,E-kerosene (PtL)
4,2028,Aviation Passenger,nachfrageniedrig-risk-neutral,8.044613e+07,e_kerosene,E-kerosene (PtL)


### Total electricity demand by scenario, all sectors

Same layout as the total-energy-demand comparison above (independent
y-scales per sector facet), but converted from fuel energy to the
electricity that fuel took to make. Drag the efficiency sliders to see how
the figure moves with the assumption; the defaults are noted in the slider
labels.

In [19]:
ptl_param = alt.param(
    name="ptl_efficiency",
    value=ELECTRICITY_SOURCE_MAP["e_kerosene"]["default_efficiency"],
    bind=alt.binding_range(min=0.10, max=1.0, step=0.01, name="PtL (e-kerosene) efficiency: "),
)
h2_param = alt.param(
    name="h2_efficiency",
    value=ELECTRICITY_SOURCE_MAP["hydrogen"]["default_efficiency"],
    bind=alt.binding_range(min=0.10, max=1.0, step=0.01, name="Electrolysis (hydrogen) efficiency: "),
)
direct_param = alt.param(
    name="direct_efficiency",
    value=ELECTRICITY_SOURCE_MAP["electricity"]["default_efficiency"],
    bind=alt.binding_range(min=0.10, max=1.0, step=0.01, name="Direct electricity efficiency: "),
)
# Vega expression, not Python: evaluated per row inside the rendered chart
# (including in a static HTML export with no live kernel), so the sliders
# above recompute the figure without re-running this cell.
_efficiency_expr = (
    "datum.source == 'e_kerosene' ? ptl_efficiency : "
    "datum.source == 'hydrogen' ? h2_efficiency : direct_efficiency"
)

electricity_compare_chart = (
    alt.Chart(electricity_fuel_df)
    .transform_calculate(efficiency=_efficiency_expr)
    .transform_calculate(electricity_twh="datum.fuel_kwh / datum.efficiency / 1000000000.0")
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("sum(electricity_twh):Q", title="Electricity demand (TWh)"),
        color=alt.Color("scenario:N", title="Scenario", scale=alt.Scale(scheme="tableau20")),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("scenario:N", title="Scenario"),
            alt.Tooltip("sum(electricity_twh):Q", title="Electricity (TWh)", format=",.3f"),
        ],
    )
    .add_params(ptl_param, h2_param, direct_param)
    .properties(width=230, height=280)
    .facet(
        column=alt.Column("sector_group:N", title=None),
        title="Electricity demand by scenario, all sectors (drag the sliders above)",
    )
    .resolve_scale(y="independent")
)
show(electricity_compare_chart)

<Vega-Lite chart>

### Which carrier the electricity goes to, per sector/case

Same PtL/hydrogen/direct-electricity split, stacked by carrier for one
Sector + Case at a time -- pick which with the dropdowns.

In [20]:
elec_sector_options = sorted(electricity_fuel_df["sector_group"].unique().tolist()) or sector_options
elec_case_options = sorted(electricity_fuel_df["scenario"].unique().tolist()) or case_options

elec_sector_dropdown = alt.binding_select(options=elec_sector_options, name="Sector: ")
elec_sector_select = alt.selection_point(
    fields=["sector_group"], bind=elec_sector_dropdown, value=[{"sector_group": elec_sector_options[0]}],
)
elec_case_dropdown = alt.binding_select(options=elec_case_options, name="Case: ")
elec_case_select = alt.selection_point(
    fields=["scenario"], bind=elec_case_dropdown, value=[{"scenario": elec_case_options[0]}],
)
elec_source_legend_select = alt.selection_point(fields=["source_label"], bind="legend")

electricity_breakdown_chart = (
    alt.Chart(electricity_fuel_df)
    .transform_calculate(efficiency=_efficiency_expr)
    .transform_calculate(electricity_twh="datum.fuel_kwh / datum.efficiency / 1000000000.0")
    .mark_area()
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("sum(electricity_twh):Q", title="Electricity demand (TWh)", stack="zero"),
        color=alt.Color("source_label:N", title="Made via", scale=alt.Scale(scheme="set2")),
        opacity=alt.condition(elec_source_legend_select, alt.value(0.9), alt.value(0.08)),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("scenario:N", title="Case"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("source_label:N", title="Made via"),
            alt.Tooltip("sum(electricity_twh):Q", title="Electricity (TWh)", format=",.3f"),
        ],
    )
    .add_params(ptl_param, h2_param, direct_param, elec_sector_select, elec_case_select, elec_source_legend_select)
    .transform_filter(elec_sector_select)
    .transform_filter(elec_case_select)
    .properties(width=760, height=380, title="Electricity demand by carrier over time")
)
show(electricity_breakdown_chart)

<Vega-Lite chart>

### Electricity demand, year x case (default efficiencies)

The chart above is interactive; this is the plain table of numbers behind
it -- one row per year, one column per case, at the default efficiencies
noted above (a static table can't follow the sliders, since those only
recompute inside the rendered chart).

In [21]:
_default_efficiency = {
    source: spec["default_efficiency"] for source, spec in ELECTRICITY_SOURCE_MAP.items()
}
electricity_default_df = electricity_fuel_df.copy()
electricity_default_df["electricity_twh"] = (
    electricity_default_df["fuel_kwh"]
    / electricity_default_df["source"].map(_default_efficiency)
    / 1_000_000_000.0
)

for _sector in sorted(electricity_default_df["sector_group"].unique()):
    _sector_totals = (
        electricity_default_df.loc[electricity_default_df["sector_group"].eq(_sector)]
        .groupby(["year", "scenario"], as_index=False)["electricity_twh"].sum()
    )
    _cells = _sector_totals.pivot(index="year", columns="scenario", values="electricity_twh").round(3)
    print(f"--- {_sector}: electricity demand (TWh) ---")
    display(_cells)

--- Aviation Cargo: electricity demand (TWh) ---


scenario,aktuellepolitiken-risk-neutral,fokush2-risk-neutral,fokusstrom-risk-neutral,hohenachfrage-risk-neutral,nachfrageniedrig-risk-neutral,technologiemix-risk-neutral,worstcasemean-risk-averse,worstcasetailrisk-risk-averse
year,,,,,,,,
2030,NaN,0.207,0.207,NaN,0.207,0.207,0.207,0.207
2031,NaN,0.206,0.206,NaN,0.206,0.206,0.206,0.206
2032,NaN,0.206,0.206,NaN,0.206,0.206,0.206,0.206
2033,NaN,0.206,0.206,NaN,0.206,0.206,0.206,0.206
2034,NaN,0.206,0.206,NaN,0.206,0.206,0.206,0.206
2035,0.237,2.128,0.474,0.237,0.474,0.474,2.128,2.128
2036,0.236,2.100,0.468,0.236,0.468,0.468,2.100,2.100
2037,0.236,2.100,0.468,0.236,0.468,0.468,2.100,2.100
2038,0.236,2.100,0.468,0.236,0.468,0.468,2.100,2.100


--- Aviation Passenger: electricity demand (TWh) ---


scenario,aktuellepolitiken-risk-neutral,fokush2-risk-neutral,fokusstrom-risk-neutral,hohenachfrage-risk-neutral,nachfrageniedrig-risk-neutral,technologiemix-risk-neutral,worstcasemean-risk-averse,worstcasetailrisk-risk-averse
year,,,,,,,,
2028,0.168,0.168,0.168,0.168,0.168,0.168,0.168,0.168
2029,0.207,0.207,0.207,0.207,0.207,0.207,0.207,0.207
2030,0.242,0.236,0.232,0.234,0.233,0.235,0.242,0.242
2031,0.281,0.274,0.270,0.271,0.271,0.273,0.281,0.281
2032,0.341,0.333,0.328,0.330,0.329,0.332,0.341,0.341
2033,0.656,0.661,0.630,0.633,0.632,0.658,0.678,0.678
2034,1.873,1.892,1.837,1.846,1.876,1.886,1.898,1.898
2035,3.388,3.063,3.061,3.038,2.927,3.169,3.545,3.545
2036,4.354,3.876,3.911,3.858,3.709,3.995,4.592,4.592


--- Maritime Cargo: electricity demand (TWh) ---


scenario,aktuellepolitiken-risk-neutral,fokush2-risk-neutral,fokusstrom-risk-neutral,hohenachfrage-risk-neutral,nachfrageniedrig-risk-neutral,technologiemix-risk-neutral,worstcasemean-risk-averse,worstcasetailrisk-risk-averse
year,,,,,,,,
2026,0.335,0.339,0.345,0.339,0.339,0.339,0.335,0.335
2027,0.222,0.226,0.229,0.225,0.230,0.226,0.222,0.222
2028,0.219,0.223,0.225,0.221,0.228,0.222,0.219,0.219
2029,0.327,0.329,0.330,0.326,0.336,0.328,0.327,0.327
2030,0.324,0.322,0.320,0.316,0.334,0.320,0.324,0.324
2031,0.514,0.525,0.522,0.517,0.543,0.527,0.514,0.514
2032,0.514,0.486,0.486,0.479,0.502,0.492,0.514,0.514
2033,1.025,0.929,0.932,0.916,0.958,0.951,1.025,1.025
2034,1.183,1.040,1.050,1.026,1.068,1.076,1.183,1.183


### Electricity demand by fuel type, year x case

The tables above give one combined total per case; this breaks that total
down by which technology the electricity is going to (e-kerosene, hydrogen,
hybrid-assist battery, ...) for a single case at a time -- set `_FUEL_TYPE_SECTOR`
and `_FUEL_TYPE_CASE` below to the sector/case you want to inspect.

In [22]:
_FUEL_TYPE_SECTOR = "Aviation Passenger"
_FUEL_TYPE_CASE = "RN–ExPol"

_fuel_type_slice = electricity_default_df.loc[
    electricity_default_df["sector_group"].eq(_FUEL_TYPE_SECTOR)
    & electricity_default_df["scenario"].eq(_FUEL_TYPE_CASE)
]
if _fuel_type_slice.empty:
    print(f"No electricity-derived carriers for {_FUEL_TYPE_SECTOR} / {_FUEL_TYPE_CASE}.")
else:
    _by_fuel_type = (
        _fuel_type_slice.pivot_table(
            index="year", columns="source_label", values="electricity_twh", aggfunc="sum",
        )
        .fillna(0.0)
        .round(3)
    )
    print(f"--- {_FUEL_TYPE_SECTOR} / {_FUEL_TYPE_CASE}: electricity demand by fuel type (TWh) ---")
    display(_by_fuel_type)

--- Aviation Passenger / aktuellepolitiken-risk-neutral: electricity demand by fuel type (TWh) ---


source_label,E-kerosene (PtL),Renewable hydrogen
year,,
2028,0.168,0.000
2029,0.207,0.000
2030,0.242,0.000
2031,0.281,0.000
2032,0.341,0.000
2033,0.656,0.000
2034,1.873,0.000
2035,3.388,0.000
2036,4.354,0.000


## Total emissions over time, all sectors and scenarios

In [23]:
emissions_by_scenario = compare_energy_df.groupby(
    ["year", "sector_group", "scenario"], as_index=False,
)["total_emission"].sum()
emissions_by_scenario["total_emission_kt"] = emissions_by_scenario["total_emission"] / 1_000_000.0

emissions_compare_chart = (
    alt.Chart(emissions_by_scenario)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("total_emission_kt:Q", title="Total emissions (kt CO2)"),
        color=alt.Color("scenario:N", title="Scenario", scale=alt.Scale(scheme="tableau20")),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("scenario:N", title="Scenario"),
            alt.Tooltip("total_emission_kt:Q", title="Emissions (kt CO2)", format=",.1f"),
        ],
    )
    .properties(width=230, height=280)
    .facet(
        column=alt.Column("sector_group:N", title=None),
        title="Total emissions by scenario, all sectors",
    )
    .resolve_scale(y="independent")
)
show(emissions_compare_chart)

<Vega-Lite chart>

## Capex spending over time, all sectors and scenarios

In [24]:
if compare_investments_df.empty:
    print("No investments CSV found for any scenario variant.")
else:
    capex_by_scenario = compare_investments_df.groupby(
        ["year", "sector_group", "scenario"], as_index=False,
    )["investment_cost_eur"].sum()
    capex_by_scenario["investment_cost_eur_m"] = capex_by_scenario["investment_cost_eur"] / 1_000_000.0

    capex_compare_chart = (
        alt.Chart(capex_by_scenario)
        .mark_line(point=True)
        .encode(
            x=alt.X("year:O", title="Year"),
            y=alt.Y("investment_cost_eur_m:Q", title="Investment (EUR millions)"),
            color=alt.Color("scenario:N", title="Scenario", scale=alt.Scale(scheme="tableau20")),
            tooltip=[
                alt.Tooltip("sector_group:N", title="Sector"),
                alt.Tooltip("year:O", title="Year"),
                alt.Tooltip("scenario:N", title="Scenario"),
                alt.Tooltip("investment_cost_eur_m:Q", title="Investment (EUR M)", format=",.1f"),
            ],
        )
        .properties(width=230, height=280)
        .facet(
            column=alt.Column("sector_group:N", title=None),
            title="Capex spending by scenario, all sectors",
        )
        .resolve_scale(y="independent")
    )
    show(capex_compare_chart)

<Vega-Lite chart>

## Final-year snapshot across all cases

A quick at-a-glance bar comparing the final simulated year's alternative
share directly across every scenario and sector, sorted from lowest to
highest within each sector panel.

In [25]:
final_year_compare = int(compare_summary["year"].max())
final_snapshot = compare_summary.loc[
    compare_summary["year"] == final_year_compare,
    ["sector_group", "scenario", "alternative_share"],
]

snapshot_chart = (
    alt.Chart(final_snapshot)
    .mark_bar()
    .encode(
        x=alt.X("alternative_share:Q", title="Alternative technology share", axis=alt.Axis(format="%")),
        y=alt.Y("scenario:N", title="Scenario", sort="-x"),
        color=alt.Color("scenario:N", title="Scenario", scale=alt.Scale(scheme="tableau20"), legend=None),
        tooltip=[
            alt.Tooltip("sector_group:N", title="Sector"),
            alt.Tooltip("scenario:N", title="Scenario"),
            alt.Tooltip("alternative_share:Q", title="Alternative share", format=".1%"),
        ],
    )
    .properties(width=220, height=max(180, 22 * final_snapshot["scenario"].nunique()))
    .facet(
        column=alt.Column("sector_group:N", title=None),
        title=f"Final-year ({final_year_compare}) alternative share by scenario, all sectors",
    )
    .resolve_scale(x="independent")
)
show(snapshot_chart)

<Vega-Lite chart>

# 15-minute electricity demand by airport/port

Loads the 15-minute-resolution electricity-demand CSVs produced by
`analysis/export_electricity_15min.py` (one gzipped CSV per sector, scenarios
stacked with a `scenario` column standing in for what would otherwise be a
sheet per scenario -- see that script's docstring for the full derivation:
real quarter-hour-of-week activity patterns from OpenSky/AIS timestamps,
applied to the electricity each run's e-kerosene/hydrogen/hybrid-assist fuel
needed, restricted to only the years each scenario actually has nonzero
demand in).

These files are large (up to ~6.5 million rows once every scenario is
stacked, well past Excel's ~1,048,576-row sheet limit) and are meant to be
read with pandas, not opened directly in a spreadsheet -- which is exactly
what this section does.

In [26]:
ELECTRICITY_15MIN_DIR = PROJECT_ROOT / "data" / "processed" / "electricity_15min"

# Which sector to load -- edit and re-run this cell to switch.
_QH_SECTOR_FILE = "cargo"  # one of: "cargo", "passenger", "maritime_cargo"

_qh_path = ELECTRICITY_15MIN_DIR / f"{_QH_SECTOR_FILE}_electricity_demand_15min.csv.gz"
assert _qh_path.exists(), (
    f"{_qh_path} not found -- run `python analysis/export_electricity_15min.py` first."
)
quarter_hourly = pd.read_csv(_qh_path, parse_dates=["timestamp"])
qh_value_columns = [c for c in quarter_hourly.columns if c not in ("scenario", "timestamp")]

print(f"Loaded {_qh_path.name}: {len(quarter_hourly):,} rows")
print(f"Scenarios present: {sorted(quarter_hourly['scenario'].unique())}")
print(f"Hub/fuel-type columns: {qh_value_columns}")
quarter_hourly.head()

Loaded cargo_electricity_demand_15min.csv.gz: 5,539,968 rows
Scenarios present: ['aktuellepolitiken-risk-neutral', 'fokush2-risk-neutral', 'fokusstrom-risk-neutral', 'hohenachfrage-risk-neutral', 'nachfrageniedrig-risk-neutral', 'technologiemix-risk-neutral', 'worstcasemean-risk-averse', 'worstcasetailrisk-risk-averse']
Hub/fuel-type columns: ['Leipzig-Halle (EDDP) -- Renewable hydrogen', 'Leipzig-Halle (EDDP) -- E-kerosene (PtL)', 'Frankfurt Am Main (EDDF) -- Renewable hydrogen']


,scenario,timestamp,Leipzig-Halle (EDDP) -- Renewable hydrogen,Leipzig-Halle (EDDP) -- E-kerosene (PtL),Frankfurt Am Main (EDDF) -- Renewable hydrogen
0,aktuellepolitiken-risk-neutral,2035-01-01 00:00:00,6784.187485,0.0,0.0
1,aktuellepolitiken-risk-neutral,2035-01-01 00:15:00,6456.478590,0.0,0.0
2,aktuellepolitiken-risk-neutral,2035-01-01 00:30:00,6413.105353,0.0,0.0
3,aktuellepolitiken-risk-neutral,2035-01-01 00:45:00,6425.810645,0.0,0.0
4,aktuellepolitiken-risk-neutral,2035-01-01 01:00:00,6367.103436,0.0,0.0


## Daily overview, one scenario

The raw data is too dense to plot directly over its full multi-year span
(hundreds of thousands of points per column) -- resampled to daily totals
here for an at-a-glance view; the next cell zooms into a real quarter-hour
window.

In [27]:
_QH_SCENARIO = sorted(quarter_hourly["scenario"].unique())[0]  # edit to any value printed above

_qh_scenario_slice = quarter_hourly.loc[quarter_hourly["scenario"].eq(_QH_SCENARIO)]
_qh_daily = (
    _qh_scenario_slice.set_index("timestamp")[qh_value_columns]
    .resample("D")
    .sum()
    .reset_index()
    .melt(id_vars="timestamp", var_name="hub_fuel_type", value_name="electricity_kwh")
)
_qh_daily = _qh_daily.loc[_qh_daily["electricity_kwh"] > 0]

qh_overview_chart = (
    alt.Chart(_qh_daily)
    .mark_line()
    .encode(
        x=alt.X("timestamp:T", title="Date"),
        y=alt.Y("electricity_kwh:Q", title="Electricity demand (kWh/day)"),
        color=alt.Color(
            "hub_fuel_type:N",
            title="Hub -- fuel type",
            scale=alt.Scale(scheme="set2"),
            legend=alt.Legend(labelLimit=280, symbolLimit=20),
        ),
        tooltip=[
            alt.Tooltip("timestamp:T", title="Date"),
            alt.Tooltip("hub_fuel_type:N", title="Hub -- fuel type"),
            alt.Tooltip("electricity_kwh:Q", title="Electricity (kWh)", format=",.0f"),
        ],
    )
    .properties(width=760, height=340, title=f"{_QH_SECTOR_FILE} / {_QH_SCENARIO}: daily electricity demand")
    .interactive()
)
show(qh_overview_chart)

<Vega-Lite chart>

## True 15-minute resolution, one week

The actual quarter-hour granularity, zoomed into one representative week
picked automatically (the first full week with nonzero demand for this
scenario) -- edit `_QH_WEEK_START` to any Monday within the scenario's
active date range to look at a different week.

In [28]:
_qh_nonzero = _qh_scenario_slice.loc[_qh_scenario_slice[qh_value_columns].sum(axis=1) > 0]
_QH_WEEK_START = (
    _qh_nonzero["timestamp"].min().normalize() - pd.Timedelta(days=_qh_nonzero["timestamp"].min().dayofweek)
)
_qh_week = _qh_scenario_slice.loc[
    _qh_scenario_slice["timestamp"].between(_QH_WEEK_START, _QH_WEEK_START + pd.Timedelta(days=7))
]
_qh_week_long = _qh_week.melt(
    id_vars="timestamp", value_vars=qh_value_columns, var_name="hub_fuel_type", value_name="electricity_kwh",
)

qh_week_chart = (
    alt.Chart(_qh_week_long)
    .mark_line()
    .encode(
        x=alt.X("timestamp:T", title="Time"),
        y=alt.Y("electricity_kwh:Q", title="Electricity demand (kWh per 15 min)"),
        color=alt.Color(
            "hub_fuel_type:N",
            title="Hub -- fuel type",
            scale=alt.Scale(scheme="set2"),
            legend=alt.Legend(labelLimit=280, symbolLimit=20),
        ),
        tooltip=[
            alt.Tooltip("timestamp:T", title="Timestamp"),
            alt.Tooltip("hub_fuel_type:N", title="Hub -- fuel type"),
            alt.Tooltip("electricity_kwh:Q", title="Electricity (kWh)", format=",.3f"),
        ],
    )
    .properties(
        width=760, height=340,
        title=f"{_QH_SECTOR_FILE} / {_QH_SCENARIO}: 15-minute demand, week of {_QH_WEEK_START.date()}",
    )
    .interactive()
)
show(qh_week_chart)

<Vega-Lite chart>

# Energy demand by technology, all 8 cases at once

The sections above already cover **total** energy (`Total energy demand
comparison`) and **total** electricity demand (`Electricity demand over
time, all sectors and scenarios`) across all 8 cases side by side. What's
still missing is the *composition* behind those totals: which fuel/
technology each case actually invests in, and how that mix evolves year by
year -- directly answering "energy demand per year based on the invested
technology in each case."

Each chart below is a small-multiples grid: one panel per case (all 8 at
once, no dropdown needed), stacked by fuel/technology family, with a
**shared y-axis across all 8 panels** so magnitude is directly comparable
at a glance. Aviation cargo and passenger share one fuel vocabulary
(kerosene / drop-in SAF / e-kerosene / hydrogen / efficient turbofan) so
their families are colored consistently between the two; maritime cargo
has its own separate vocabulary and chart.

Segment/size suffixes (`_medium`, `_long`, `_regional`, `_deepsea`, ...) are
stripped to a "technology family" for coloring -- otherwise aviation
passenger alone would need 12 colors and maritime 14, past what a
categorical palette can keep reliably distinguishable (the dataviz skill's
rule: fold past ~8 series to "Other" or facet, never generate a 9th hue).
One deliberate exception: maritime's `methanol_ready`/`ammonia_ready`
technologies are grouped into the `marine_diesel` family rather than kept
separate -- per this catalog's own technology_notes, a "ready" vessel still
burns 100% conventional fuel today (the capex premium buys a cheaper
*future* conversion, not present-day usage), so labeling its actual energy
demand as marine_diesel reflects what it really burns, not its catalog
name.

In [29]:
import re

_SEGMENT_SUFFIX_PATTERN = re.compile(r"_(regional|deepsea|short|medium|long)$")
_FREIGHT_SUFFIX_PATTERN = re.compile(r"_freight$")


def _technology_family(technology_name: str) -> str:
    family = _SEGMENT_SUFFIX_PATTERN.sub("", str(technology_name))
    # Aviation cargo technologies are named "<fuel>_freight_<size>"
    # (e.g. "kerosene_freight_medium") -- passenger's are just
    # "<fuel>_<size>", with no "_freight" infix. Strip it after the size
    # suffix so cargo and passenger share the same family name ("kerosene",
    # not "kerosene_freight") and can use one consistent color scale.
    family = _FREIGHT_SUFFIX_PATTERN.sub("", family)
    # "Ready" vessels (methanol_ready, ammonia_ready) still burn 100%
    # conventional marine diesel today -- per this catalog's own
    # technology_notes, "ready" only buys a cheaper future conversion, it
    # does not mean the fuel is already in use. Grouping with marine_diesel
    # reflects what they actually burn, not their catalog name.
    if family.endswith("_ready"):
        return "marine_diesel"
    return family


family_energy = energy.copy()
family_energy["family"] = family_energy["current_technology"].map(_technology_family)
family_energy["energy_kwh_total"] = pd.to_numeric(
    family_energy.get("primary_energy_consumption", 0.0), errors="coerce",
).fillna(0.0) + pd.to_numeric(
    family_energy.get("secondary_energy_consumption", 0.0), errors="coerce",
).fillna(0.0)

energy_by_family = family_energy.groupby(
    ["sector_group", "scenario", "year", "family"], as_index=False,
)["energy_kwh_total"].sum()
energy_by_family["energy_twh"] = energy_by_family["energy_kwh_total"] / 1_000_000_000.0

print("Technology families found, per sector:")
for _sector in sorted(energy_by_family["sector_group"].unique()):
    _families = sorted(
        energy_by_family.loc[energy_by_family["sector_group"].eq(_sector), "family"].unique(),
    )
    print(f"  {_sector}: {_families}")

Technology families found, per sector:
  Aviation Cargo: ['drop_in_saf', 'e_kerosene', 'hydrogen', 'kerosene']
  Aviation Passenger: ['drop_in_saf', 'e_kerosene', 'efficient_turbofan', 'hydrogen', 'kerosene']
  Maritime Cargo: ['ammonia', 'biofuel', 'hybrid', 'lng', 'marine_diesel', 'methanol']


In [30]:
# One fixed hue per technology family, shared between the aviation
# cargo/passenger charts (same fuel vocabulary) so a family is always the
# same color across both. Duplicated from analysis/cartographic_dashboard.py
# (CATEGORICAL_PALETTE), not imported -- this notebook is meant to stand
# alone, same convention already used for ELECTRICITY_SOURCE_MAP above.
# Validated with the dataviz skill's palette validator for stacked-area use.
CATEGORICAL_PALETTE = (
    "#2a78d6",  # blue
    "#eb6834",  # orange
    "#1baf7a",  # aqua
    "#eda100",  # yellow
    "#e87ba4",  # magenta
    "#008300",  # green
    "#4a3aa7",  # violet
    "#e34948",  # red
)

AVIATION_FAMILY_ORDER = ["kerosene", "drop_in_saf", "e_kerosene", "hydrogen", "efficient_turbofan"]
MARITIME_FAMILY_ORDER = sorted(
    energy_by_family.loc[energy_by_family["sector_group"].eq("Maritime Cargo"), "family"].unique(),
)
assert len(MARITIME_FAMILY_ORDER) <= len(CATEGORICAL_PALETTE), (
    f"{len(MARITIME_FAMILY_ORDER)} maritime families needs folding to Other -- "
    f"only {len(CATEGORICAL_PALETTE)} validated colors available."
)

AVIATION_FAMILY_SCALE = alt.Scale(
    domain=AVIATION_FAMILY_ORDER, range=list(CATEGORICAL_PALETTE[: len(AVIATION_FAMILY_ORDER)]),
)
MARITIME_FAMILY_SCALE = alt.Scale(
    domain=MARITIME_FAMILY_ORDER, range=list(CATEGORICAL_PALETTE[: len(MARITIME_FAMILY_ORDER)]),
)


def _family_stack_chart(sector_name: str, family_order: list[str], family_scale: alt.Scale) -> alt.Chart:
    data = energy_by_family.loc[energy_by_family["sector_group"].eq(sector_name)]
    return (
        alt.Chart(data)
        .mark_area()
        .encode(
            x=alt.X("year:O", title="Year"),
            y=alt.Y("energy_twh:Q", title="Energy demand (TWh)", stack="zero"),
            color=alt.Color(
                "family:N", title="Technology family", scale=family_scale, sort=family_order,
            ),
            tooltip=[
                alt.Tooltip("scenario:N", title="Case"),
                alt.Tooltip("year:O", title="Year"),
                alt.Tooltip("family:N", title="Technology family"),
                alt.Tooltip("energy_twh:Q", title="Energy (TWh)", format=",.3f"),
            ],
        )
        .properties(width=170, height=220)
        .facet(
            column=alt.Column(
                "scenario:N", title=None, header=alt.Header(labelAngle=-40, labelAlign="right"),
            ),
            title=f"{sector_name}: energy demand by technology family, all 8 cases",
        )
    )

## Aviation cargo: energy demand by technology family, all 8 cases

In [31]:
show(_family_stack_chart("Aviation Cargo", AVIATION_FAMILY_ORDER, AVIATION_FAMILY_SCALE))

<Vega-Lite chart>

**Key findings, aviation cargo:**

- Total energy demand is nearly identical across all 8 cases (~0.40 TWh in
  2025 on average) -- underlying freight demand doesn't vary by scenario
  (confirmed earlier: `freight_tonne_km_demand` is scenario-invariant in
  this dataset), so what differs between cases is *which fuel* meets that
  demand, not how much fuel is needed.
- Every case crosses 50% alternative-fuel share by 2035 -- but that
  crossover is driven almost entirely by drop-in (biofuel) SAF, not
  e-kerosene: e-kerosene's own onset year varies sharply by case (2035 for
  fokusH2, vs. 2043-2047 for the rest), which the stacked composition above
  makes visible even though the *aggregate* crossover date looks uniform.
- By 2050, drop-in SAF supplies the majority of energy in every case
  (55.1% of the 2050 total, summed across all 8), with e-kerosene a modest
  minority contributor (11.6%) and plain kerosene down to a third (33.3%).
- Total energy demand itself grows **+89.4%** from 2025 to 2050 -- that
  growth comes from rising freight demand, not from the fuel switch.

## Aviation passenger: energy demand by technology family, all 8 cases

In [32]:
show(_family_stack_chart("Aviation Passenger", AVIATION_FAMILY_ORDER, AVIATION_FAMILY_SCALE))

<Vega-Lite chart>

**Key findings, aviation passenger:**

- Alternative-fuel share crosses 50% later than cargo (2038-2040 depending
  on case) but with less spread across cases. The two risk-averse cases
  (worstcasemean/worstcasetailrisk) cross earliest, at 2038 -- consistent
  with the earlier finding that risk-averse decision logic hedges into
  e-kerosene sooner than risk-neutral logic even under identical prices.
- By 2050, drop-in SAF and e-kerosene together supply **79.0%** of total
  energy (46.9% + 32.1%), with e-kerosene a much larger share of the mix
  here than in cargo (32.1% vs. cargo's 11.6%) -- passenger's much larger
  fleet gives e-kerosene far more investment opportunities to win its
  near-tied competition against biofuel SAF (see the earlier NPV
  investigation: the two are close enough that fleet size, not economics
  alone, determines how often each one gets picked).
- The two risk-averse cases reach the highest final alternative share of
  all 8 (99.2% non-conventional by 2050, vs. 85.9-91.6% for the six
  risk-neutral cases) -- direct evidence of the robust-decision framework
  hedging harder against downside fuel-price/carbon-price risk than a
  risk-neutral view would.
- Total energy demand grows **+22.7%** (2025 to 2050) -- between cargo's
  higher growth and maritime's near-flat trend.

## Maritime cargo: energy demand by technology family, all 8 cases

Maritime's `methanol_regional` family is real e-methanol/methanol dual-fuel
use (not a "ready" vessel), so it keeps its own color -- only the
`_ready` technologies were folded into `marine_diesel` above.

In [33]:
show(_family_stack_chart("Maritime Cargo", MARITIME_FAMILY_ORDER, MARITIME_FAMILY_SCALE))

<Vega-Lite chart>

**Key findings, maritime cargo:**

- `marine_diesel` collapses from the dominant fuel in 2025 (65.8% share,
  summed across all 8 cases) to a minor residual by 2050 (5.7%) -- vessels
  replace rather than retrofit, so this reflects real fleet turnover as
  ships reach their replacement year, not fuel-switching within existing
  hulls.
- `biofuel`, which starts as the *second*-largest source in 2025 (34.2%,
  reflecting the 6 real EU-MRV-verified biofuel_deepsea vessels merged into
  this fleet), nearly disappears by 2050 -- `hybrid` and `lng` take over as
  the dominant pathways instead, together supplying **90.6%** of the 2050
  total.
- All 8 cases cross 50% alternative-fuel share remarkably early and
  uniformly, by **2031** -- much earlier than either aviation sector,
  reflecting maritime's already-substantial real-world biofuel baseline
  plus a comparatively easier mandate to satisfy early on.
- Total energy demand stays almost flat (**+2.9%**, 2025 to 2050) -- for
  maritime cargo, this transition is almost entirely about *which* fuel is
  used, not how much energy the fleet needs.

# Passenger-km demand, all 8 cases

`passenger_km_demand` is **exogenous input** to the model, not a simulation
output -- it's read straight from each case's `aviation_scenario.csv`, the
same way the model itself computes it (`allocated_passenger_km` in
`aviation_passenger_airline.py`): total national demand per segment
(short/medium/long) times each airline's `operator_market_share`. Neither
input varies by year in this dataset (checked directly: every operator's
share is byte-identical across 2025-2050), so an airline's absolute volume
here scales exactly with total demand -- its *share* of the market never
shifts on its own.

Because `passenger_km_demand` is keyed by `scenario_id` (the belief
scenario), not by case folder, this reads each case's own
`ambiguity_aware_decision.realized_scenario_id` from its `scenario.yaml`
first -- the same case-to-belief-scenario mapping already established
earlier in this notebook/session (all 6 risk-neutral cases realize their
own matching scenario; both risk-averse cases realize `aktuellepolitiken`).

In [34]:
import yaml

PASSENGER_INPUT_ROOT = PROJECT_ROOT / "data" / "input"
PASSENGER_CASE_PREFIX = "baseline-passenger-transition"


def _realized_scenario_id(case_folder: str) -> str:
    config = yaml.safe_load((PASSENGER_INPUT_ROOT / case_folder / "scenario.yaml").read_text())
    return config["ambiguity_aware_decision"]["realized_scenario_id"]


_pkm_total_rows = []
_pkm_operator_rows = []

for variant in SCENARIO_VARIANTS:
    case_folder = f"{PASSENGER_CASE_PREFIX}-{variant}"
    scenario_csv = PASSENGER_INPUT_ROOT / case_folder / "aviation_scenario.csv"
    if not scenario_csv.exists():
        continue
    belief_scenario = _realized_scenario_id(case_folder)
    scen = pd.read_csv(scenario_csv)
    year_columns = [c for c in scen.columns if c.isdigit()]

    demand = scen.loc[
        (scen["variable_name"] == "passenger_km_demand") & (scen["scenario_id"] == belief_scenario),
        ["segment", *year_columns],
    ].set_index("segment")
    share = scen.loc[scen["variable_name"] == "operator_market_share", ["operator_name", "segment", *year_columns]]

    for year in year_columns:
        segment_demand = demand[year].astype(float)
        _pkm_total_rows.append(
            {"scenario": case_label(variant), "year": int(year), "passenger_km": segment_demand.sum()},
        )
        operator_pkm = share.assign(
            passenger_km=share.apply(
                lambda row, year=year: float(row[year]) * segment_demand.get(row["segment"], 0.0), axis=1,
            ),
        )
        for operator_name, total in operator_pkm.groupby("operator_name")["passenger_km"].sum().items():
            _pkm_operator_rows.append(
                {"scenario": case_label(variant), "year": int(year), "operator_name": operator_name, "passenger_km": total},
            )

passenger_km_total = pd.DataFrame(_pkm_total_rows)
passenger_km_total["billion_pkm"] = passenger_km_total["passenger_km"] / 1e9

passenger_km_by_operator = pd.DataFrame(_pkm_operator_rows)
passenger_km_by_operator["billion_pkm"] = passenger_km_by_operator["passenger_km"] / 1e9

print(f"{len(passenger_km_total):,} case-year rows, {len(passenger_km_by_operator):,} case-year-operator rows")
passenger_km_total.head()

208 case-year rows, 2,288 case-year-operator rows


,scenario,year,passenger_km,billion_pkm
0,aktuellepolitiken-risk-neutral,2025,1.420165e+11,142.016535
1,aktuellepolitiken-risk-neutral,2026,1.420165e+11,142.016535
2,aktuellepolitiken-risk-neutral,2027,1.420165e+11,142.016535
3,aktuellepolitiken-risk-neutral,2028,1.420165e+11,142.016535
4,aktuellepolitiken-risk-neutral,2029,1.420165e+11,142.016535


## Total passenger-km per year, all 8 cases

In [35]:
pkm_total_chart = (
    alt.Chart(passenger_km_total)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("billion_pkm:Q", title="Passenger-km (billions)"),
        color=alt.Color("scenario:N", title="Case", scale=alt.Scale(scheme="tableau20")),
        tooltip=[
            alt.Tooltip("scenario:N", title="Case"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("billion_pkm:Q", title="Passenger-km (billions)", format=",.2f"),
        ],
    )
    .properties(width=640, height=320, title="Total passenger-km demand per year, all 8 cases")
)
show(pkm_total_chart)

<Vega-Lite chart>

**Key findings:**

- Demand *declines* in every one of the 8 cases -- by 2050 it's down
  2.5% (the three aktuellepolitiken-realized cases: aktuellepolitiken-
  risk-neutral, worstcasemean, worstcasetailrisk) to as much as 19.3%
  (nachfrageniedrig). This is an exogenous input the model reacts to, not
  something the technology transition itself changes.
- The scenario names describe energy-system-wide assumptions (electricity/
  hydrogen focus, demand growth), not aviation-specific passenger demand
  directly -- confirmed here: `hohenachfrage` ("high demand") shows a
  *larger* decline (-16.2%) than `fokusstrom` (-15.6%), so its name should
  not be read as "this case has the highest passenger-km."
- Because `passenger_km_demand` differs by `scenario_id` while the mapping
  from case to `scenario_id` is many-to-one (both risk-averse cases realize
  `aktuellepolitiken`), those two cases' lines sit exactly on top of the
  `aktuellepolitiken-risk-neutral` line here -- the three cases share
  identical demand, and differ only in which technology they invest in
  (see the energy-demand-by-technology section above).

## Passenger-km by airline, all 8 cases

Top 7 airlines by volume kept distinct; the remaining 4 (each under 0.14
billion pkm/year -- LEAV Aviation, Sundair, Arcus-Air, Universal Sky
Carrier) are folded into "Other" per the dataviz skill's rule (fold past
~8 series rather than generate more hues).

In [36]:
_operator_totals = (
    passenger_km_by_operator.groupby("operator_name")["billion_pkm"].sum().sort_values(ascending=False)
)
TOP_OPERATORS = _operator_totals.index[:7].tolist()

pkm_by_operator_grouped = passenger_km_by_operator.copy()
pkm_by_operator_grouped["operator_group"] = pkm_by_operator_grouped["operator_name"].where(
    pkm_by_operator_grouped["operator_name"].isin(TOP_OPERATORS), "Other",
)
pkm_by_operator_grouped = pkm_by_operator_grouped.groupby(
    ["scenario", "year", "operator_group"], as_index=False,
)["billion_pkm"].sum()

OPERATOR_ORDER = [*TOP_OPERATORS, "Other"]
OPERATOR_SCALE = alt.Scale(
    domain=OPERATOR_ORDER, range=[*CATEGORICAL_PALETTE[: len(TOP_OPERATORS)], "#9c9a93"],
)

pkm_operator_chart = (
    alt.Chart(pkm_by_operator_grouped)
    .mark_area()
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("billion_pkm:Q", title="Passenger-km (billions)", stack="zero"),
        color=alt.Color("operator_group:N", title="Airline", scale=OPERATOR_SCALE, sort=OPERATOR_ORDER),
        tooltip=[
            alt.Tooltip("scenario:N", title="Case"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("operator_group:N", title="Airline"),
            alt.Tooltip("billion_pkm:Q", title="Passenger-km (billions)", format=",.3f"),
        ],
    )
    .properties(width=170, height=220)
    .facet(
        column=alt.Column(
            "scenario:N", title=None, header=alt.Header(labelAngle=-40, labelAlign="right"),
        ),
        title="Passenger-km by airline, all 8 cases",
    )
)
show(pkm_operator_chart)

<Vega-Lite chart>

**Key findings:**

- Lufthansa alone carries roughly 70% of total passenger-km in every case
  (its market share is fixed and dominant across all three segments --
  82.9% of long-haul, 68.9% of medium-haul, 39.7% of short-haul) --
  visually, the stack is Lufthansa-dominated in every one of the 8 panels.
- Since market share never changes over time in this dataset, every
  airline's band grows in *exact proportion* to total demand -- the stack
  composition looks identical across years within a case; only its overall
  height changes. Any visual difference between the 8 panels comes purely
  from that case's underlying demand growth, not from airlines gaining or
  losing share.

## Passenger-km by domestic vs. international, all 8 cases

**Correction from the first version of this section**: it originally
reported the fleet's full theoretical *capacity* split (seats x occupancy
x real distance) as if all of it were flown. But we already established
that capacity exceeds the exogenous `passenger_km_demand` in every case,
every year -- so reporting raw capacity overstates what's actually served.

This version instead reports the **exogenous demand actually covered**
(the same `passenger_km_demand x operator_market_share` used in the total/
by-airline charts above -- capped at capacity in the impossible-in-
practice case demand ever exceeds it), split into domestic/international
using the real fleet's own domestic/international *ratio* as the
allocation key (per operator x segment x year, from `aircraft.csv`'s
`domestic_activity_share_base`/`international_activity_share_base` --
weighted by each aircraft's own capacity share of its operator+segment).
The ratio comes from real data; the magnitude it's applied to is the
demand actually being served, not idle capacity.

In [37]:
AIRCRAFT_GEO_USECOLS = [
    "year", "operator_name", "current_technology", "segment",
    "domestic_activity_share_base", "international_activity_share_base", "annual_distance_km_base",
]

_geo2_rows = []
_geo2_operator_rows = []

for variant in SCENARIO_VARIANTS:
    case_folder = f"{PASSENGER_CASE_PREFIX}-{variant}"
    case_dir = RESULTS_ROOT / case_folder
    ac_path = case_dir / "aircraft.csv"
    if not ac_path.exists():
        continue
    belief_scenario = _realized_scenario_id(case_folder)

    header = pd.read_csv(ac_path, nrows=0).columns
    ac = pd.read_csv(ac_path, usecols=[c for c in AIRCRAFT_GEO_USECOLS if c in header])
    catalog = pd.read_csv(PASSENGER_INPUT_ROOT / case_folder / "aviation_technology_catalog.csv").set_index(
        "technology_name",
    )[["economy_seats", "business_seats", "first_class_seats"]]
    scen = pd.read_csv(PASSENGER_INPUT_ROOT / case_folder / "aviation_scenario.csv")
    year_columns = [c for c in scen.columns if c.isdigit()]

    occupancy = {}
    for cabin, variable in [("economy", "economy_occupancy"), ("business", "business_occupancy"), ("first", "first_occupancy")]:
        rows = scen.loc[scen["variable_name"] == variable, ["operator_name", *year_columns]]
        occupancy[cabin] = rows.set_index("operator_name")

    ac = ac.join(catalog, on="current_technology")
    for cabin, seat_col in [("economy", "economy_seats"), ("business", "business_seats"), ("first", "first_class_seats")]:
        occ_table = occupancy[cabin]
        ac[f"{cabin}_occupancy"] = ac.apply(
            lambda row, occ_table=occ_table: float(occ_table.loc[row["operator_name"], str(int(row["year"]))])
            if row["operator_name"] in occ_table.index
            else 0.0,
            axis=1,
        )
    ac["capacity_weighted_seats"] = (
        ac["economy_seats"] * ac["economy_occupancy"]
        + ac["business_seats"] * ac["business_occupancy"]
        + ac["first_class_seats"] * ac["first_occupancy"]
    )
    ac["passenger_km_capacity"] = ac["capacity_weighted_seats"] * ac["annual_distance_km_base"].fillna(0.0)
    ac["domestic_pkm_capacity"] = ac["passenger_km_capacity"] * ac["domestic_activity_share_base"].fillna(0.0)
    ac["international_pkm_capacity"] = ac["passenger_km_capacity"] * ac["international_activity_share_base"].fillna(0.0)

    capacity_by_op_seg_year = ac.groupby(["year", "operator_name", "segment"], as_index=False)[
        ["domestic_pkm_capacity", "international_pkm_capacity"]
    ].sum()
    capacity_by_op_seg_year["total_capacity"] = (
        capacity_by_op_seg_year["domestic_pkm_capacity"] + capacity_by_op_seg_year["international_pkm_capacity"]
    )
    # Real fleet's own domestic/international ratio -- this is the allocation
    # key, applied below to demand actually covered, not to raw capacity.
    capacity_by_op_seg_year["domestic_ratio"] = (
        capacity_by_op_seg_year["domestic_pkm_capacity"] / capacity_by_op_seg_year["total_capacity"]
    ).fillna(0.0)

    demand = scen.loc[
        (scen["variable_name"] == "passenger_km_demand") & (scen["scenario_id"] == belief_scenario),
        ["segment", *year_columns],
    ].set_index("segment")
    share = scen.loc[
        scen["variable_name"] == "operator_market_share", ["operator_name", "segment", *year_columns],
    ]

    demand_rows = []
    for year in year_columns:
        segment_demand = demand[year].astype(float)
        op_demand = share.assign(
            demand_covered=share.apply(
                lambda row, year=year: float(row[year]) * segment_demand.get(row["segment"], 0.0), axis=1,
            ),
        )[["operator_name", "segment", "demand_covered"]]
        op_demand["year"] = int(year)
        demand_rows.append(op_demand)
    demand_by_op_seg_year = pd.concat(demand_rows, ignore_index=True)

    merged = demand_by_op_seg_year.merge(
        capacity_by_op_seg_year[["year", "operator_name", "segment", "domestic_ratio", "total_capacity"]],
        on=["year", "operator_name", "segment"],
        how="left",
    )
    merged["domestic_ratio"] = merged["domestic_ratio"].fillna(0.0)
    # No capacity cap here: the model itself never truncates demand by
    # capacity -- a shortfall only ever triggers *growth* (more capacity),
    # it never reduces the demand figure. Capping would silently understate
    # 2025 specifically (the static initial fleet, before year 2026's first
    # growth step runs, has real per-operator/segment shortfalls in a few
    # cases -- e.g. Lufthansa short-haul: 9.91B capacity vs 11.82B demand in
    # 2025 alone) and break reconciliation with the demand-side total above.
    merged["domestic_actual_pkm"] = merged["demand_covered"] * merged["domestic_ratio"]
    merged["international_actual_pkm"] = merged["demand_covered"] * (1.0 - merged["domestic_ratio"])
    merged["scenario"] = case_label(variant)

    _geo2_rows.append(
        merged.groupby(["scenario", "year"], as_index=False)[
            ["domestic_actual_pkm", "international_actual_pkm"]
        ].sum(),
    )
    _geo2_operator_rows.append(
        merged.groupby(["scenario", "year", "operator_name"], as_index=False)[
            ["domestic_actual_pkm", "international_actual_pkm"]
        ].sum(),
    )

passenger_km_geo = pd.concat(_geo2_rows, ignore_index=True)
passenger_km_geo["domestic_billion_pkm"] = passenger_km_geo["domestic_actual_pkm"] / 1e9
passenger_km_geo["international_billion_pkm"] = passenger_km_geo["international_actual_pkm"] / 1e9

passenger_km_geo_by_operator = pd.concat(_geo2_operator_rows, ignore_index=True)
passenger_km_geo_by_operator["domestic_billion_pkm"] = passenger_km_geo_by_operator["domestic_actual_pkm"] / 1e9
passenger_km_geo_by_operator["international_billion_pkm"] = (
    passenger_km_geo_by_operator["international_actual_pkm"] / 1e9
)

print(f"{len(passenger_km_geo):,} case-year rows, {len(passenger_km_geo_by_operator):,} case-year-operator rows")
passenger_km_geo.head()

208 case-year rows, 2,288 case-year-operator rows


,scenario,year,domestic_actual_pkm,international_actual_pkm,domestic_billion_pkm,international_billion_pkm
0,aktuellepolitiken-risk-neutral,2025,1.178200e+10,1.302345e+11,11.781995,130.234540
1,aktuellepolitiken-risk-neutral,2026,1.058585e+10,1.314307e+11,10.585851,131.430684
2,aktuellepolitiken-risk-neutral,2027,1.057103e+10,1.314455e+11,10.571033,131.445502
3,aktuellepolitiken-risk-neutral,2028,1.057538e+10,1.314412e+11,10.575379,131.441156
4,aktuellepolitiken-risk-neutral,2029,1.057538e+10,1.314412e+11,10.575379,131.441156


In [38]:
domestic_chart = (
    alt.Chart(passenger_km_geo)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("domestic_billion_pkm:Q", title="Domestic passenger-km (billions)"),
        color=alt.Color("scenario:N", title="Case", scale=alt.Scale(scheme="tableau20")),
        tooltip=[
            alt.Tooltip("scenario:N", title="Case"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("domestic_billion_pkm:Q", title="Domestic passenger-km (billions)", format=",.3f"),
        ],
    )
    .properties(width=640, height=320, title="Domestic passenger-km actually covered per year, all 8 cases")
)
show(domestic_chart)

<Vega-Lite chart>

In [39]:
international_chart = (
    alt.Chart(passenger_km_geo)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("international_billion_pkm:Q", title="International passenger-km (billions)"),
        color=alt.Color("scenario:N", title="Case", scale=alt.Scale(scheme="tableau20")),
        tooltip=[
            alt.Tooltip("scenario:N", title="Case"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("international_billion_pkm:Q", title="International passenger-km (billions)", format=",.3f"),
        ],
    )
    .properties(width=640, height=320, title="International passenger-km actually covered per year, all 8 cases")
)
show(international_chart)

<Vega-Lite chart>

**Key findings:**

- Domestic + international now sum exactly back to the demand-side total
  passenger-km chart shown earlier in this notebook (by construction --
  same `demand_covered`, just split by the real fleet's domestic/
  international ratio), unlike the earlier capacity-based version which
  didn't reconcile with it.
- The domestic/international *ratio* itself still comes from real fleet
  data, so the split still reflects genuine route-network structure -- only
  the total each is applied to changed, from capacity to demand covered.

## Passenger-km actually covered by domestic/international, split by airline

Same top-7-airlines-plus-"Other" grouping and color scale as the earlier
"Passenger-km by airline" section, applied to demand actually covered
(not capacity) domestic and international totals.

In [40]:
pkm_geo_by_operator_grouped = passenger_km_geo_by_operator.copy()
pkm_geo_by_operator_grouped["operator_group"] = pkm_geo_by_operator_grouped["operator_name"].where(
    pkm_geo_by_operator_grouped["operator_name"].isin(TOP_OPERATORS), "Other",
)
pkm_geo_by_operator_grouped = pkm_geo_by_operator_grouped.groupby(
    ["scenario", "year", "operator_group"], as_index=False,
)[["domestic_billion_pkm", "international_billion_pkm"]].sum()


def _geo_operator_chart(value_field: str, title: str) -> alt.Chart:
    return (
        alt.Chart(pkm_geo_by_operator_grouped)
        .mark_area()
        .encode(
            x=alt.X("year:O", title="Year"),
            y=alt.Y(f"{value_field}:Q", title="Passenger-km (billions)", stack="zero"),
            color=alt.Color("operator_group:N", title="Airline", scale=OPERATOR_SCALE, sort=OPERATOR_ORDER),
            tooltip=[
                alt.Tooltip("scenario:N", title="Case"),
                alt.Tooltip("year:O", title="Year"),
                alt.Tooltip("operator_group:N", title="Airline"),
                alt.Tooltip(f"{value_field}:Q", title="Passenger-km (billions)", format=",.3f"),
            ],
        )
        .properties(width=170, height=220)
        .facet(
            column=alt.Column(
                "scenario:N", title=None, header=alt.Header(labelAngle=-40, labelAlign="right"),
            ),
            title=title,
        )
    )

In [41]:
show(_geo_operator_chart("domestic_billion_pkm", "Domestic passenger-km actually covered, by airline, all 8 cases"))

<Vega-Lite chart>

In [42]:
show(_geo_operator_chart("international_billion_pkm", "International passenger-km actually covered, by airline, all 8 cases"))

<Vega-Lite chart>

**Key findings:**

- Lufthansa's dominance is real in both splits but notably stronger
  internationally: it holds **68.9%** of the international market vs.
  **51.1%** of the domestic market (2025) -- still the largest carrier
  domestically, just with more competition there than abroad. (These
  shares are unchanged from the capacity-based version -- they come from
  the ratio, which didn't change; only the totals they're applied to did.)
- `Lufthansa CityLine` is the clearest domestic-leaning carrier: 27.9% of
  its own passenger-km is domestic, well above Lufthansa's own 11.2% and
  the fleet-wide norm. `Arcus-Air` and `Universal Sky Carrier` remain more
  internationally skewed than Lufthansa itself (domestic shares of 11.1%
  and 4.7%) -- being small does not mean being domestic-focused in this
  fleet.

# Corrected passenger-km demand: bottom-up anchor + real trend

Two separate problems were found with the exogenous `passenger_km_demand`
used in every chart above:
1. It's frozen flat from 2035 to 2050 (no real data past 2035, just the
   2035 value repeated) -- confirmed by comparing it against a real,
   externally-sourced demand table for the same 6 scenarios.
2. Its absolute scale is uncertain -- it doesn't match either of the two
   real reference statistics found (Germany's own airport-traffic total,
   or German-registered-airlines' worldwide network total), and its
   scope/derivation was never traceable in this repository.

**Fix**: anchor the 2025 starting point to the bottom-up, real fleet
capacity already computed above (seats x occupancy x each real aircraft's
actual annual distance, from OpenSky data) -- since that figure is fully
traceable and doesn't depend on any external/uncertain scenario input.
Then apply the *shape* (not the absolute level) of the externally-sourced
demand table's year-over-year evolution on top of that anchor. Growth
rates are scale-independent, so this sidesteps the open scale/category
question entirely -- the external table's absolute values are never used,
only its ratio to its own 2025 value:

`corrected_demand(year) = bottom_up_2025 x [external_source(year) / external_source(2025)]`

The external source table has real, distinct values at 6 milestone years
(2025/2030/2035/2040/2045/2050, no freezing) for both domestic and
international, for all 6 belief scenarios -- linearly interpolated here to
get all 26 years, since 5-year spacing is tight enough that a straight
line between real points is a reasonable read of the true path (unlike the
old step-function extension, which had to stretch a single point across
up to 15 flat years).

In [43]:
import numpy as np

# The user's externally-sourced demand table (billion pkm/yr), Germany,
# both aviation demand categories -- entered here as given, at its own 6
# real milestone years. Only ITS RATIOS are used below, never these
# absolute values directly (see markdown above for why).
EXTERNAL_SOURCE_YEARS = [2025, 2030, 2035, 2040, 2045, 2050]
EXTERNAL_DOMESTIC = {
    "aktuellepolitiken": [7.71587, 8.238908, 8.465852, 7.601238, 6.999996, 6.902289],
    "fokusstrom":        [7.345665, 6.30344, 4.36277, 2.795592, 2.284706, 2.305169],
    "fokusH2":           [7.39483, 6.361581, 4.163284, 2.436318, 1.847063, 1.765963],
    "technologiemix":    [7.428984, 6.673811, 4.842491, 3.168151, 2.58683, 2.450347],
    "hohenachfrage":     [7.077492, 5.786817, 3.911551, 2.416489, 1.9505, 2.116354],
    "niedrigenachfrage": [8.030147, 7.72641, 6.179655, 4.331726, 3.683577, 2.989801],
}
EXTERNAL_INTERNATIONAL = {
    "aktuellepolitiken": [200.0018, 195.3572, 194.347, 191.67, 193.3258, 193.0225],
    "fokusstrom":        [202.8452, 190.3615, 172.7145, 155.6097, 150.1633, 157.8021],
    "fokusH2":           [201.85, 192.622, 169.5142, 149.2466, 142.24, 145.1224],
    "technologiemix":    [202.6688, 192.4463, 176.9608, 159.8539, 153.3259, 161.0819],
    "hohenachfrage":     [200.873, 189.6586, 169.8437, 151.7742, 144.6291, 147.9022],
    "niedrigenachfrage": [191.1842, 180.1212, 154.5391, 134.7325, 123.1345, 119.9678],
}

# Bottom-up 2025 anchor -- the real fleet capacity total already computed
# earlier in this notebook (seats x occupancy x real annual distance),
# now using each registration real per-flight-route domestic/international
# split (data/archive/registration_matching/fix_domestic_international_split.py)
# instead of the old generic aircraft-type-level average.
BOTTOM_UP_DOMESTIC_2025 = 10_832_817_710.44
BOTTOM_UP_INTERNATIONAL_2025 = 121_755_144_231.31

ALL_YEARS = list(range(2025, 2051))

# Each case's realized belief scenario (same mapping established earlier:
# risk-neutral cases match their own name, both risk-averse cases realize
# aktuellepolitiken).
CASE_TO_BELIEF_SCENARIO = {
    "aktuellepolitiken-risk-neutral": "aktuellepolitiken",
    "fokush2-risk-neutral": "fokusH2",
    "fokusstrom-risk-neutral": "fokusstrom",
    "hohenachfrage-risk-neutral": "hohenachfrage",
    "nachfrageniedrig-risk-neutral": "niedrigenachfrage",
    "technologiemix-risk-neutral": "technologiemix",
    "worstcasemean-risk-averse": "aktuellepolitiken",
    "worstcasetailrisk-risk-averse": "aktuellepolitiken",
}

_corrected_rows = []
for case, belief_scenario in CASE_TO_BELIEF_SCENARIO.items():
    dom_interp = np.interp(ALL_YEARS, EXTERNAL_SOURCE_YEARS, EXTERNAL_DOMESTIC[belief_scenario])
    intl_interp = np.interp(ALL_YEARS, EXTERNAL_SOURCE_YEARS, EXTERNAL_INTERNATIONAL[belief_scenario])
    dom_ratio = dom_interp / dom_interp[0]
    intl_ratio = intl_interp / intl_interp[0]
    for year, dr, ir in zip(ALL_YEARS, dom_ratio, intl_ratio):
        _corrected_rows.append(
            {
                "scenario": case_label(case),
                "year": year,
                "domestic_pkm": BOTTOM_UP_DOMESTIC_2025 * dr,
                "international_pkm": BOTTOM_UP_INTERNATIONAL_2025 * ir,
            },
        )

corrected_demand = pd.DataFrame(_corrected_rows)
corrected_demand["total_pkm"] = corrected_demand["domestic_pkm"] + corrected_demand["international_pkm"]
corrected_demand["domestic_billion_pkm"] = corrected_demand["domestic_pkm"] / 1e9
corrected_demand["international_billion_pkm"] = corrected_demand["international_pkm"] / 1e9
corrected_demand["total_billion_pkm"] = corrected_demand["total_pkm"] / 1e9

print(f"{len(corrected_demand):,} case-year rows")
corrected_demand.head()

208 case-year rows


,scenario,year,domestic_pkm,international_pkm,total_pkm,domestic_billion_pkm,international_billion_pkm,total_billion_pkm
0,aktuellepolitiken-risk-neutral,2025,1.083282e+10,1.217551e+11,1.325880e+11,10.832818,121.755144,132.587962
1,aktuellepolitiken-risk-neutral,2026,1.097968e+10,1.211896e+11,1.321693e+11,10.979683,121.189645,132.169329
2,aktuellepolitiken-risk-neutral,2027,1.112655e+10,1.206241e+11,1.317507e+11,11.126549,120.624147,131.750695
3,aktuellepolitiken-risk-neutral,2028,1.127341e+10,1.200586e+11,1.313321e+11,11.273414,120.058648,131.332062
4,aktuellepolitiken-risk-neutral,2029,1.142028e+10,1.194931e+11,1.309134e+11,11.420280,119.493149,130.913429


## Corrected total passenger-km per year, all 8 cases

In [44]:
corrected_total_chart = (
    alt.Chart(corrected_demand)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("total_billion_pkm:Q", title="Passenger-km (billions)"),
        color=alt.Color("scenario:N", title="Case", scale=alt.Scale(scheme="tableau20")),
        tooltip=[
            alt.Tooltip("scenario:N", title="Case"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("total_billion_pkm:Q", title="Passenger-km (billions)", format=",.3f"),
        ],
    )
    .properties(width=640, height=320, title="Corrected total passenger-km per year, all 8 cases")
)
show(corrected_total_chart)

<Vega-Lite chart>

## Corrected domestic vs. international passenger-km per year

In [45]:
corrected_domestic_chart = (
    alt.Chart(corrected_demand)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("domestic_billion_pkm:Q", title="Domestic passenger-km (billions)"),
        color=alt.Color("scenario:N", title="Case", scale=alt.Scale(scheme="tableau20")),
        tooltip=[
            alt.Tooltip("scenario:N", title="Case"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("domestic_billion_pkm:Q", title="Domestic passenger-km (billions)", format=",.3f"),
        ],
    )
    .properties(width=640, height=320, title="Corrected domestic passenger-km per year, all 8 cases")
)
show(corrected_domestic_chart)

<Vega-Lite chart>

In [46]:
corrected_international_chart = (
    alt.Chart(corrected_demand)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("international_billion_pkm:Q", title="International passenger-km (billions)"),
        color=alt.Color("scenario:N", title="Case", scale=alt.Scale(scheme="tableau20")),
        tooltip=[
            alt.Tooltip("scenario:N", title="Case"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("international_billion_pkm:Q", title="International passenger-km (billions)", format=",.3f"),
        ],
    )
    .properties(width=640, height=320, title="Corrected international passenger-km per year, all 8 cases")
)
show(corrected_international_chart)

<Vega-Lite chart>

**Key findings:**

- Every case now starts at exactly the same, real, bottom-up 2025 anchor
  (19.50B domestic + 113.09B international = 132.59B total) -- unlike the
  raw exogenous demand chart earlier, which started all 8 cases at an
  unverified 142.02B.
- The trajectory now genuinely evolves through 2050 instead of freezing at
  2035 -- e.g. niedrigenachfrage domestic falls to roughly a third of its
  2025 anchor by 2050 (following the external source's own steep decline
  ratio), where the old frozen version would have shown no change at all
  after 2035.
- This is explicitly a **shape-only correction**: the absolute scale still
  reflects this model's own fleet (bottom-up, real, but scoped to only the
  ~11 tracked German airlines), not either external reference statistic's
  scale. If the scale question is resolved later (Way 1 vs. Way 2), only
  the anchor constants above need to change -- the interpolation/ratio
  logic stays the same.

# Corrected aviation cargo demand: bottom-up anchor + researched trend

Same problem as passenger demand, same fix. `freight_tonne_km_demand` is
confirmed identical across all 6 belief scenarios (checked directly --
this is intentional: no scenario-specific cargo forecast exists, so a
single researched estimate is used for all cases) but it freezes flat from
2035 to 2050, just like the passenger data did.

No cargo-specific research was already saved in this project (checked the
archive first, as requested), so this section is built from external
research, same standard as the passenger fix:
- **Near-term real growth rates**: BALM (Bundesamt für Logistik und
  Mobilität) / Intraplan Consult "Gleitende Mittelfristprognose Sommer
  2025" -- Germany's official freight-transport medium-term forecast --
  gives +1.1% (2026) and +1.7% (2027) for German air freight volume.
- **Long-term trend**: Boeing's Commercial Market Outlook / World Air
  Cargo Forecast -- the standard industry reference for multi-decade air
  cargo growth -- projects ~3.7%/year through 2045 (~4%/year, "traffic to
  double," through 2043 in Boeing's other headline figure; the more
  conservative 3.7% is used here).

**Anchor**: the bottom-up 2025 figure (payload capacity x load factor x
each real aircraft's actual annual distance, same formula as the
passenger/maritime bottom-up calculations), not the model's own 2025
scenario value -- consistent with how the passenger fix was anchored.
Interestingly, cargo's bottom-up capacity (5.79 billion tonne-km) is
*below* the model's own 2025 demand figure (8.39 billion) -- the opposite
of passenger, where capacity exceeded demand. This is directionally
sensible: air cargo demand has a real structural growth story (e-commerce,
emerging-market trade), unlike passenger demand, which declines in every
one of the 8 cases in this dataset.

In [47]:
BOTTOM_UP_CARGO_2025 = 5_788_100_000.0  # bottom-up 2025 anchor (billion tonne-km x 1e9)

_cargo_growth_factor = {2025: 1.0}
_cargo_growth_factor[2026] = _cargo_growth_factor[2025] * 1.011  # BALM real 2026 forecast: +1.1%
_cargo_growth_factor[2027] = _cargo_growth_factor[2026] * 1.017  # BALM real 2027 forecast: +1.7%
BOEING_LONG_TERM_CARGO_GROWTH_RATE = 1.037  # Boeing Commercial Market Outlook: ~3.7%/yr through 2045
for _year in range(2028, 2051):
    _cargo_growth_factor[_year] = _cargo_growth_factor[_year - 1] * BOEING_LONG_TERM_CARGO_GROWTH_RATE

CARGO_CASE_NAMES = [
    "aktuellepolitiken-risk-neutral", "fokush2-risk-neutral", "fokusstrom-risk-neutral",
    "hohenachfrage-risk-neutral", "nachfrageniedrig-risk-neutral", "technologiemix-risk-neutral",
    "worstcasemean-risk-averse", "worstcasetailrisk-risk-averse",
]

_cargo_rows = []
for case in CARGO_CASE_NAMES:
    for year, factor in _cargo_growth_factor.items():
        _cargo_rows.append(
            {"scenario": case_label(case), "year": year, "freight_tonne_km": BOTTOM_UP_CARGO_2025 * factor},
        )

corrected_cargo_demand = pd.DataFrame(_cargo_rows)
corrected_cargo_demand["billion_tonne_km"] = corrected_cargo_demand["freight_tonne_km"] / 1e9

print(f"{len(corrected_cargo_demand):,} case-year rows")
print(f"2025: {corrected_cargo_demand.loc[corrected_cargo_demand.year==2025, 'billion_tonne_km'].iloc[0]:.3f} billion tonne-km")
print(f"2050: {corrected_cargo_demand.loc[corrected_cargo_demand.year==2050, 'billion_tonne_km'].iloc[0]:.3f} billion tonne-km")
corrected_cargo_demand.head()

208 case-year rows
2025: 5.788 billion tonne-km
2050: 13.725 billion tonne-km


,scenario,year,freight_tonne_km,billion_tonne_km
0,aktuellepolitiken-risk-neutral,2025,5.788100e+09,5.788100
1,aktuellepolitiken-risk-neutral,2026,5.851769e+09,5.851769
2,aktuellepolitiken-risk-neutral,2027,5.951249e+09,5.951249
3,aktuellepolitiken-risk-neutral,2028,6.171445e+09,6.171445
4,aktuellepolitiken-risk-neutral,2029,6.399789e+09,6.399789


## Corrected cargo freight-tonne-km per year, all 8 cases

In [48]:
corrected_cargo_chart = (
    alt.Chart(corrected_cargo_demand)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("billion_tonne_km:Q", title="Freight tonne-km (billions)"),
        color=alt.Color("scenario:N", title="Case", scale=alt.Scale(scheme="tableau20")),
        tooltip=[
            alt.Tooltip("scenario:N", title="Case"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("billion_tonne_km:Q", title="Freight tonne-km (billions)", format=",.3f"),
        ],
    )
    .properties(width=640, height=320, title="Corrected aviation cargo demand per year, all 8 cases")
)
show(corrected_cargo_chart)

<Vega-Lite chart>

**Key findings:**

- All 8 cases show exactly one overlapping line -- as expected, since cargo
  demand is a single researched estimate, not scenario-differentiated (this
  is the correct, confirmed behavior, not a bug).
- The corrected trajectory grows smoothly from **5.79 billion tonne-km**
  (2025 bottom-up anchor) to **13.73 billion tonne-km** (2050) -- a 2.37x
  increase, replacing the old behavior of freezing flat at the 2035 value
  for the last 15 years.
- This anchor is *below* the model's own previous 2025 demand figure
  (8.39B) -- i.e. this correction lowers near-term demand relative to what
  was there before, while raising it in the long run relative to the old
  frozen tail. Both directions matter: the model previously overstated
  near-term demand relative to real fleet capacity, and understated
  long-term demand relative to the real industry growth outlook.

# Corrected maritime cargo demand: bottom-up anchor + researched trend

Same shape of fix as aviation, one important difference: **our maritime
cargo fleet is 100% international by construction** -- checked directly,
`domestic_activity_share_base` is exactly 0.0 for all 131 real vessels, so
there's no domestic/international split to build here. The only valid
comparison is against the source table's **International Shipping** row;
its "Domestic" row almost certainly represents German inland-waterway/
coastal barge traffic, a vessel population this model doesn't track at
all.

**Vessel-type coverage caveat** (the "maritime is tricky" point): this
model only tracks 5 real vessel types (Container Ship, General Cargo Ship,
Oil Products Tanker, Heavy Load Carrier, Chemical/Oil Products Tanker),
not the full universe the external source table covers (which also
includes bulk carriers, LNG/gas carriers, RoRo/vehicle carriers, ferries,
etc.). Checked directly: our bottom-up total covers a remarkably stable
~35% of the source table's international total across all 6 scenarios
(34.4%-35.6%) -- that stability is a good sign the gap is a real,
consistent vessel-type-coverage effect, not noise. Container ships alone
are 87.7% of our bottom-up total, plausible for Germany specifically
(German shipowners are historically world-leading in container tonnage,
much more so than in bulk carriers).

**Method** (identical to the aviation passenger fix): anchor 2025 to the
bottom-up total (396.8 billion tonne-km, real EU-MRV/AIS-derived vessel
data), then apply each scenario's *shape* (not absolute level) from the
external source table's international-shipping row, linearly interpolated
between its 6 real milestone years (2025/2030/2035/2040/2045/2050).

In [49]:
import numpy as np

MARITIME_SOURCE_YEARS = [2025, 2030, 2035, 2040, 2045, 2050]
MARITIME_INTERNATIONAL_SOURCE = {
    "aktuellepolitiken": [1132.589, 1085.835, 1083.662, 1072.627, 1088.106, 1093.808],
    "fokusstrom":        [1152.703, 1050.792, 941.419, 835.69, 802.8498, 846.5119],
    "fokusH2":           [1147.746, 1069.873, 925.2986, 800.4855, 754.0194, 772.8511],
    "technologiemix":    [1150.407, 1066.186, 970.5334, 865.7197, 825.5345, 871.3598],
    "hohenachfrage":     [1124.674, 1030.89, 896.1981, 788.8556, 747.0533, 760.8934],
    "niedrigenachfrage": [1113.531, 1078.43, 918.9942, 811.096, 760.4907, 778.4789],
}

# Bottom-up 2025 anchor -- real EU-MRV/AIS-derived fleet capacity, 100%
# international (domestic_activity_share_base confirmed 0.0 for all 131
# real vessels), so this is the international-only figure directly.
BOTTOM_UP_MARITIME_CARGO_2025 = 396_823_000_000.0

MARITIME_CASE_TO_BELIEF_SCENARIO = {
    "aktuellepolitiken-risk-neutral": "aktuellepolitiken",
    "fokush2-risk-neutral": "fokusH2",
    "fokusstrom-risk-neutral": "fokusstrom",
    "hohenachfrage-risk-neutral": "hohenachfrage",
    "nachfrageniedrig-risk-neutral": "niedrigenachfrage",
    "technologiemix-risk-neutral": "technologiemix",
    "worstcasemean-risk-averse": "aktuellepolitiken",
    "worstcasetailrisk-risk-averse": "aktuellepolitiken",
}

ALL_YEARS_MARITIME = list(range(2025, 2051))

_maritime_rows = []
for case, belief_scenario in MARITIME_CASE_TO_BELIEF_SCENARIO.items():
    interp = np.interp(ALL_YEARS_MARITIME, MARITIME_SOURCE_YEARS, MARITIME_INTERNATIONAL_SOURCE[belief_scenario])
    ratio = interp / interp[0]
    for year, r in zip(ALL_YEARS_MARITIME, ratio):
        _maritime_rows.append(
            {"scenario": case_label(case), "year": year, "freight_tonne_km": BOTTOM_UP_MARITIME_CARGO_2025 * r},
        )

corrected_maritime_demand = pd.DataFrame(_maritime_rows)
corrected_maritime_demand["billion_tonne_km"] = corrected_maritime_demand["freight_tonne_km"] / 1e9

print(f"{len(corrected_maritime_demand):,} case-year rows")
corrected_maritime_demand.head()

208 case-year rows


,scenario,year,freight_tonne_km,billion_tonne_km
0,aktuellepolitiken-risk-neutral,2025,3.968230e+11,396.823000
1,aktuellepolitiken-risk-neutral,2026,3.935468e+11,393.546778
2,aktuellepolitiken-risk-neutral,2027,3.902706e+11,390.270557
3,aktuellepolitiken-risk-neutral,2028,3.869943e+11,386.994335
4,aktuellepolitiken-risk-neutral,2029,3.837181e+11,383.718114


## Corrected maritime cargo freight-tonne-km per year, all 8 cases

In [50]:
corrected_maritime_chart = (
    alt.Chart(corrected_maritime_demand)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("billion_tonne_km:Q", title="Freight tonne-km (billions)"),
        color=alt.Color("scenario:N", title="Case", scale=alt.Scale(scheme="tableau20")),
        tooltip=[
            alt.Tooltip("scenario:N", title="Case"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("billion_tonne_km:Q", title="Freight tonne-km (billions)", format=",.2f"),
        ],
    )
    .properties(width=640, height=320, title="Corrected maritime cargo demand per year, all 8 cases")
)
show(corrected_maritime_chart)

<Vega-Lite chart>

**Key findings:**

- Every case starts at the same real, bottom-up 2025 anchor (396.8 billion
  tonne-km) and then diverges according to its own scenario's real
  international-shipping trend -- unlike aviation cargo (one flat
  researched estimate for all scenarios), maritime genuinely has distinct
  per-scenario source data, so the 8 lines separate visibly after 2025.
- Both risk-averse cases (worstcasemean, worstcasetailrisk) track
  aktuellepolitiken's trend exactly, same realized-scenario mapping used
  throughout this notebook -- aktuellepolitiken is also the mildest
  decline of the six, dipping only to ~1072-1093B (source scale) rather
  than the ~750-950B range most other scenarios fall to by mid-century.
- This remains anchored to *our* fleet's real scale (396.8B, ~35% of the
  source table's total), not the source table's own absolute level -- if
  the vessel-type coverage gap is resolved later (e.g. by adding bulk
  carriers to the tracked fleet), only the anchor constant needs to
  change, not this interpolation/ratio logic.

# Technology adoption and final energy by carrier, all 8 cases

Two views per sector, laid out like the steel-sector slides:

1. **Adoption over time** -- share of installed capacity by technology family, one panel per case
   (RN = risk-neutral, RA = risk-averse). Capacity = nominal seats (passenger) or payload (cargo,
   maritime) x each asset's real annual distance, active assets only.
2. **Final energy by carrier** -- the fuel actually consumed (TWh per year) in 2030-2050, one bar per case.
   The first bar, **No transition**, is the Existing Policies fleet's total energy in that year
   assigned entirely to the fossil carrier -- a reference for how much energy is *switched*, not a
   separate simulation run (the model has no run with investment disabled).
   Bars below it in later years mostly reflect each scenario's lower demand (fleet utilization
   follows scenario demand), not energy saved by switching fuels.
   In both charts aviation passenger and cargo are shown as one unit. Energy is summed; for the
   adoption shares, seats and payload tonnes cannot be added, so aviation is weighted by each
   aircraft's energy need at its real annual distance (maritime stays payload-weighted).

In [51]:
CASE_DISPLAY = dict(CASE_LABEL)  # paper labels, keyed by functional case id
CASE_ORDER = list(CASE_DISPLAY.values())
NO_TRANSITION_LABEL = "No transition"
INPUT_ROOT = PROJECT_ROOT / "data" / "input"

# prefix, technology catalog file, capacity basis
SECTOR_CAPACITY_BASIS = {
    "Aviation Passenger": ("baseline-passenger-transition", "aviation_technology_catalog.csv", "seats"),
    "Aviation Cargo": ("baseline-cargo-transition", "aviation_technology_catalog.csv", "payload"),
    "Maritime Cargo": ("baseline-maritime-cargo-transition", "maritime_technology_catalog.csv", "payload"),
}

_capacity_rows = []
for _sector, (_prefix, _catalog_file, _basis) in SECTOR_CAPACITY_BASIS.items():
    for _variant in SCENARIO_VARIANTS:
        _folder = f"{_prefix}-{_variant}"
        _ac_path = RESULTS_ROOT / _folder / "aircraft.csv"
        if not _ac_path.exists():
            continue
        _assets = pd.read_csv(
            _ac_path, usecols=["year", "status", "current_technology", "annual_distance_km_base"],
        )
        _assets = _assets.loc[_assets["status"].astype(str).str.strip().str.lower().eq("active")].copy()
        _catalog = pd.read_csv(INPUT_ROOT / _folder / _catalog_file).set_index("technology_name")
        if _basis == "seats":
            _nominal = _catalog["economy_seats"] + _catalog["business_seats"] + _catalog["first_class_seats"]
        else:
            _nominal = _catalog["payload_capacity_kg"] / 1000.0
        _assets["capacity"] = (
            _assets["current_technology"].map(_nominal.astype(float)).fillna(0.0)
            * _assets["annual_distance_km_base"].fillna(0.0)
        )
        _assets["energy_capacity"] = (
            _assets["annual_distance_km_base"].fillna(0.0)
            / _assets["current_technology"].map(_catalog["kilometer_per_kwh"].astype(float))
        )
        _assets["family"] = _assets["current_technology"].map(_technology_family)
        _block = _assets.groupby(["year", "family"], as_index=False)[["capacity", "energy_capacity"]].sum()
        _block["sector_group"] = _sector
        _block["case_label"] = CASE_DISPLAY[_variant]
        _capacity_rows.append(_block)
adoption_capacity = pd.concat(_capacity_rows, ignore_index=True)

CARRIER_LABELS = {
    "kerosene": "kerosene (fossil)", "saf": "SAF", "e_kerosene": "e-kerosene", "hydrogen": "hydrogen",
    "marine_diesel": "marine diesel", "biofuel": "biofuel", "lng": "LNG", "methanol": "methanol",
    "ammonia": "ammonia", "electricity": "electricity",
}
AVIATION_CARRIER_ORDER = ["kerosene (fossil)", "SAF", "e-kerosene", "hydrogen"]
MARITIME_CARRIER_ORDER = ["marine diesel", "biofuel", "LNG", "methanol", "ammonia", "electricity"]
FOSSIL_CARRIER = {
    "Aviation Passenger": "kerosene (fossil)", "Aviation Cargo": "kerosene (fossil)",
    "Maritime Cargo": "marine diesel",
}

_parts = []
for _carrier_col, _energy_col in (
    ("primary_energy_carrier", "primary_energy_consumption"),
    ("secondary_energy_carrier", "secondary_energy_consumption"),
):
    _part = compare_energy_df[["sector_group", "scenario", "year", _carrier_col, _energy_col]].copy()
    _part.columns = ["sector_group", "scenario", "year", "carrier", "energy_kwh"]
    _parts.append(_part)
carrier_energy = pd.concat(_parts, ignore_index=True)
carrier_energy["carrier"] = carrier_energy["carrier"].astype(str).str.strip().str.lower()
carrier_energy = carrier_energy.loc[~carrier_energy["carrier"].isin(["none", "nan", ""])]
carrier_energy["energy_kwh"] = pd.to_numeric(carrier_energy["energy_kwh"], errors="coerce").fillna(0.0)
carrier_energy = carrier_energy.groupby(["sector_group", "scenario", "year", "carrier"], as_index=False)["energy_kwh"].sum()
carrier_energy["carrier_label"] = carrier_energy["carrier"].map(CARRIER_LABELS)
assert carrier_energy["carrier_label"].notna().all(), carrier_energy.loc[carrier_energy["carrier_label"].isna(), "carrier"].unique()
carrier_energy["case_label"] = carrier_energy["scenario"]
carrier_energy["energy_twh"] = carrier_energy["energy_kwh"] / 1e9

# "No transition" reference bar: the AktuellePolitiken fleet's total energy that year, all fossil.
_reference = (
    carrier_energy.loc[carrier_energy["scenario"].eq("RN–ExPol")]
    .groupby(["sector_group", "year"], as_index=False)["energy_twh"].sum()
)
_reference["carrier_label"] = _reference["sector_group"].map(FOSSIL_CARRIER)
_reference["case_label"] = NO_TRANSITION_LABEL
carrier_energy = pd.concat(
    [carrier_energy[["sector_group", "year", "carrier_label", "case_label", "energy_twh"]], _reference],
    ignore_index=True,
)
print(adoption_capacity["sector_group"].value_counts().to_dict())
carrier_energy.groupby(["sector_group", "carrier_label"])["energy_twh"].sum().round(0)

{'Maritime Cargo': 1184, 'Aviation Passenger': 790, 'Aviation Cargo': 641}


sector_group        carrier_label    
Aviation Cargo      SAF                    573.0
                    e-kerosene              55.0
                    hydrogen                46.0
                    kerosene (fossil)     4413.0
Aviation Passenger  SAF                    665.0
                    e-kerosene             752.0
                    hydrogen                 9.0
                    kerosene (fossil)    10028.0
Maritime Cargo      LNG                    681.0
                    ammonia                  1.0
                    biofuel                 51.0
                    electricity            217.0
                    marine diesel         2453.0
                    methanol                10.0
Name: energy_twh, dtype: float64

In [52]:
CARRIER_YEARS = [2030, 2035, 2040, 2045, 2050]


def _adoption_chart(
    sector_name: str, family_order: list[str], family_scale: alt.Scale,
    source: pd.DataFrame | None = None, y_title: str = "Share of capacity",
) -> alt.Chart:
    frame = adoption_capacity if source is None else source
    data = frame.loc[frame["sector_group"].eq(sector_name)]
    unexpected = set(data["family"].unique()) - set(family_order)
    assert not unexpected, f"{sector_name}: families without a color: {unexpected}"
    return (
        alt.Chart(data)
        .mark_area()
        .encode(
            x=alt.X(
                "year:Q", title=None,
                scale=alt.Scale(domain=[2025, 2050], nice=False),
                axis=alt.Axis(format="d", values=[2025, 2030, 2035, 2040, 2045, 2050], labelAngle=0),
            ),
            y=alt.Y("capacity:Q", stack="normalize", title=y_title, axis=alt.Axis(format="%")),
            color=alt.Color(
                "family:N", title="Technology family", scale=family_scale, sort=family_order,
            ),
            order=alt.Order("family:N", sort="ascending"),
            tooltip=[
                alt.Tooltip("case_label:N", title="Case"),
                alt.Tooltip("year:Q", title="Year", format="d"),
                alt.Tooltip("family:N", title="Technology family"),
            ],
        )
        .properties(width=210, height=140)
        .facet(
            facet=alt.Facet("case_label:N", sort=CASE_ORDER, title=None),
            columns=4,
            title=f"{sector_name}: share of capacity by technology",
        )
    )


def _carrier_chart(
    sector_name: str, carrier_order: list[str], colors: list[str] | None = None,
    source: pd.DataFrame | None = None,
) -> alt.Chart:
    frame = carrier_energy if source is None else source
    data = frame.loc[
        frame["sector_group"].eq(sector_name) & frame["year"].isin(CARRIER_YEARS)
    ].copy()
    unexpected = set(data["carrier_label"].unique()) - set(carrier_order)
    assert not unexpected, f"{sector_name}: carriers without a color: {unexpected}"
    data["carrier_rank"] = data["carrier_label"].map({c: i for i, c in enumerate(carrier_order)})
    scale = alt.Scale(domain=carrier_order, range=colors or list(CATEGORICAL_PALETTE[: len(carrier_order)]))
    return (
        alt.Chart(data)
        .mark_bar()
        .encode(
            x=alt.X(
                "case_label:N", title=None, sort=[NO_TRANSITION_LABEL, *CASE_ORDER],
                axis=alt.Axis(labelAngle=-60),
            ),
            y=alt.Y("energy_twh:Q", title="Final energy, TWh per year"),
            color=alt.Color("carrier_label:N", title="Energy carrier", scale=scale, sort=carrier_order),
            order=alt.Order("carrier_rank:Q", sort="ascending"),
            tooltip=[
                alt.Tooltip("case_label:N", title="Case"),
                alt.Tooltip("year:O", title="Year"),
                alt.Tooltip("carrier_label:N", title="Carrier"),
                alt.Tooltip("energy_twh:Q", title="TWh", format=",.2f"),
            ],
        )
        .properties(width=150, height=230)
        .facet(
            column=alt.Column("year:O", title=None),
            title=f"{sector_name}: final energy by carrier",
        )
    )

## Adoption over time: aviation (passenger + cargo) and maritime

In [53]:
# Each sector keeps its own legend: the sectors use different technologies, and a shared
# legend would need more distinct colors than the validated palette has.
# Aviation passenger and cargo as one unit. Seats and tonnes of payload cannot be added, so the
# combined aviation panels weight each aircraft by the energy it needs to fly its real annual
# distance (distance / kilometer_per_kwh), a unit both share.
_aviation_adoption = adoption_capacity.loc[
    adoption_capacity["sector_group"].isin(["Aviation Passenger", "Aviation Cargo"])
].copy()
_aviation_adoption["sector_group"] = "Aviation (passenger + cargo)"
_aviation_adoption = _aviation_adoption.groupby(
    ["sector_group", "case_label", "year", "family"], as_index=False,
)["energy_capacity"].sum().rename(columns={"energy_capacity": "capacity"})
_adoption_all = alt.vconcat(
    _adoption_chart(
        "Aviation (passenger + cargo)", AVIATION_FAMILY_ORDER, AVIATION_FAMILY_SCALE,
        source=_aviation_adoption, y_title="Share of fleet energy need",
    ),
    _adoption_chart("Maritime Cargo", MARITIME_FAMILY_ORDER, MARITIME_FAMILY_SCALE),
).resolve_scale(color="independent")
show(_adoption_all)

<Vega-Lite chart>

## Final energy by carrier: aviation (passenger + cargo) and maritime

In [54]:
# Same fuel, same color as in the adoption chart above (a carrier takes its technology
# family's color); electricity has no family of its own, so it takes the first unused palette color.
_family_color = dict(zip(MARITIME_FAMILY_ORDER, CATEGORICAL_PALETTE))
_maritime_carrier_colors = [
    _family_color["marine_diesel"], _family_color["biofuel"], _family_color["lng"],
    _family_color["methanol"], _family_color["ammonia"], CATEGORICAL_PALETTE[len(MARITIME_FAMILY_ORDER)],
]
# Aviation passenger and cargo as one unit: TWh of the same carrier simply add up (the
# "No transition" reference bar adds up the same way).
_aviation_unit = carrier_energy.loc[carrier_energy["sector_group"].isin(["Aviation Passenger", "Aviation Cargo"])].copy()
_aviation_unit["sector_group"] = "Aviation (passenger + cargo)"
_aviation_unit = _aviation_unit.groupby(
    ["sector_group", "year", "carrier_label", "case_label"], as_index=False,
)["energy_twh"].sum()
_carrier_all = alt.vconcat(
    _carrier_chart("Aviation (passenger + cargo)", AVIATION_CARRIER_ORDER, source=_aviation_unit),
    _carrier_chart("Maritime Cargo", MARITIME_CARRIER_ORDER, _maritime_carrier_colors),
).resolve_scale(color="independent")
show(_carrier_all)

<Vega-Lite chart>